<font size=10>**NETWORK - LISBON**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

<font color='#BFD72' size=5>**RESEARCH QUESTION**: </font><font size=5>*Which companies have a dominant position in public procurement?*</font> 

**Data**: 
- [*Portal BASE*](https://www.base.gov.pt/Base4/pt/pesquisa/?type=contratos&texto=&adjudicante=&adjudicataria=&tipo=2&tipocontrato=0&cpv=&aqinfo=&desdeprazoexecucao=&ateprazoexecucao=&sel_price=price_11&desdeprecocontrato=&ateprecocontrato=&desdeprecoefectivo=&ateprecoefectivo=&sel_date=date_11&desdedatacontrato=2023-01-01&atedatacontrato=2026-03-31&desdedatapublicacao=&atedatapublicacao=&desdedatafecho=&atedatafecho=&pais=0&distrito=0&concelho=0)

- [*Treated Datasets*](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/#/resources)

**Group B**
- Beatriz Marques 20231605
- Maria Inês Santos 20231630
- Luís Soeiro 20211536
- Rodrigo Silva 20231602

<font color='#BFD72F' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [1. Imports](#1)
- [2. Data Integration](#2)
- [3. The Network](#3)
  - [3.1. How is public procurement distributed?](#3.1)
  - [3.2. How highly concentrated is the network?](#3.2)
  - [3.3. Which are the dominant companies of the network?](#3.3)
  - [3.4. Which are the dominant public entities of the network?](#3.4)
  - [3.5. Why are companies specialized?](#3.5)
- [4. Random Reference Models](#4)
  - [4.1. Generate random reference models](#4.1)
- [5. Community Detection & Modularity](#5)
  - [5.1. Bipartite projection — company co-contracting network](#5.1)
  - [5.2. Company projection (co-contracting network)](#5.2)
  - [5.3. Louvain community detection](#5.3)
  - [5.4. Modularity vs. random reference](#5.4)
  - [5.5. Community size distribution](#5.5)
  - [5.6. Community characterization](#5.6)
  - [5.7. Inter-community connections & bridge companies](#5.7)
  - [5.8. Inter-community heatmap](#5.8)
- [6. Conclusions & Key Findings](#6)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
%load_ext autoreload
%autoreload 2

Failed to read module file 'c:\Users\Rafael\AppData\Local\Programs\Python\Python312\Lib\urllib\parse.py' for module 'urllib.parse': UnicodeDecodeError
Traceback (most recent call last):
  File "C:\Users\Rafael\AppData\Roaming\Python\Python312\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Rafael\AppData\Roaming\Python\Python312\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rafael\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen im

In [2]:
import importlib
import os
import subprocess
import sys
import warnings
from collections import Counter
from datetime import datetime
from itertools import combinations

import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import powerlaw
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

# Optional dependencies
for package, import_name in [("openpyxl", "openpyxl"), ("python-louvain", "community")]:
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        importlib.invalidate_caches()

import community as community_louvain
import openpyxl

In [3]:
# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

The analysis is built on a solid, complete dataset of **public contracts**, each linking a public buyer to a supplier company, with full information on prices, sectors (CPV), cities and dates. This breadth means the conclusions reflect the real market, not a narrow sample.

In [4]:
data = pd.read_csv('../data/preprocessed_data.csv')
# data.head()
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 16989 entries, 0 to 16988
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   idcontrato                   16989 non-null  int64  
 1   tipoContrato                 16989 non-null  str    
 2   tipoFimContrato              16972 non-null  str    
 3   CPV                          16989 non-null  str    
 4   adjudicante                  16989 non-null  str    
 5   adjudicatarios               16989 non-null  str    
 6   concorrentes                 14707 non-null  str    
 7   precoBaseProcedimento        16989 non-null  float64
 8   precoContratual              16989 non-null  float64
 9   PrecoTotalEfetivo            16989 non-null  float64
 10  dataDecisaoAdjudicacao       16989 non-null  str    
 11  dataCelebracaoContrato       16989 non-null  str    
 12  dataPublicacao               16989 non-null  str    
 13  dataFechoContrato          

In [5]:
data["dataPublicacao"] = pd.to_datetime(data["dataPublicacao"], errors='coerce')
data["dataCelebracaoContrato"] = pd.to_datetime(data["dataCelebracaoContrato"], errors='coerce')
data["dataDecisaoAdjudicacao"] = pd.to_datetime(data["dataDecisaoAdjudicacao"], errors='coerce')
data["dataFechoContrato"] = pd.to_datetime(data["dataFechoContrato"], errors='coerce')

In [6]:
lisbon_municipalities = [
    'Lisboa', 'Cascais', 'Amadora', 'Oeiras', 'Sintra', 'Loures',
    'Torres Vedras', 'Mafra', 'Odivelas', 'Vila Franca de Xira',
    'Alenquer', 'Lourinhã', 'Cadaval', 'Azambuja',
    'Arruda dos Vinhos', 'Sobral de Monte Agraço'
]

data = data[data['city'].isin(lisbon_municipalities)]

# <font color='#BFD72F' size=6>**3. The Network**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font size=5>**3.1. How is Public Procurement Distributed?**</font> <a class="anchor" id="3.1"></a>

[Back to TOC](#toc)

In [7]:
def compute_weight(prices, mode="log_sum"):

    prices = np.array(prices)

    if mode == "sum_price":
        return prices.sum()

    elif mode == "avg_price":
        return prices.mean()

    elif mode == "log_sum":
        return np.log(prices).sum()

    elif mode == "log_mean":
        return np.log(prices).mean()

    else:
        raise ValueError(
            f"Unknown mode: {mode}. "
            f"Valid modes: sum_price, avg_price, log_sum, log_mean"
        )

In [8]:
def _ensure_dataframe(data):
    if isinstance(data, pd.DataFrame):
        return data.copy()

    if isinstance(data, dict):
        # CASE 1: dict of lists (valid dataframe)
        try:
            df = pd.DataFrame(data)
            return df
        except Exception:
            pass

        # CASE 2: dict of scalars → wrap into list
        return pd.DataFrame([data])

    raise TypeError(f"Unsupported input type: {type(data)}")

In [9]:
def most_common(x):
    if isinstance(x, list) and len(x) > 0:
        return Counter(x).most_common(1)[0][0]
    return "Outros / Não classificado"

### <font color='#BFD72F' size=5>3.1.1 Building the Network</font> <a class="anchor" id="3.1.1"></a>

**Bipartite Network** (Companies ↔ Public Entities)

| Graph Type | Edges Represent |
|---|---|
| Multigraph | Individual contracts awarded |
| `Directed Graph` | `Grouped contracts between 2 unique entities` |

[Back to TOC](#toc)

In [10]:
def build_contract_network(data, weight_mode="log_sum") -> nx.DiGraph:
    
    df = _ensure_dataframe(data)

    # --- CLEAN COLUMNS ---
    df = df.rename(columns={
        'precoContratual': 'price',
        'adjudicante_clean': 'source',
        'adjudicatarios_clean': 'target'
    })

    # --- BASIC CLEANING ---
    df = df.dropna(subset=['source', 'target', 'price'])
    df = df[df['price'] > 0]

    # --- CONCORRENTES ---
    if 'nr_concorrentes' in df.columns:
        df['concorrentes'] = df['nr_concorrentes']
    else:
        df['concorrentes'] = pd.to_numeric(df.get('concorrentes', 0), errors='coerce').fillna(0)

    # --- EDGE AGGREGATION ---
    edge_df = (
        df.groupby(['source', 'target'], as_index=False)
        .agg(
            total_price=('price', 'sum'),
            nr_concorrentes=('concorrentes', 'sum'),
            contracts=('price', 'count'),
            price_series=('price', list),
            idcontrato=('idcontrato', list),
            tipoContrato=('tipoContrato', list),
            tipoFimContrato=('tipoFimContrato', list),
            CPV=('CPV', list),
            precoBaseProcedimento=('precoBaseProcedimento', list),
            precoContratual=('price', list),
            PrecoTotalEfetivo=('PrecoTotalEfetivo', list),
            dataDecisaoAdjudicacao=('dataDecisaoAdjudicacao', list),
            dataCelebracaoContrato=('dataCelebracaoContrato', list),
            dataPublicacao=('dataPublicacao', list),
            dataFechoContrato=('dataFechoContrato', list),
            nr_concorrentes_list=('nr_concorrentes', list),
            contribuinte_adjudicante=('contribuinte_adjudicante', list),
            contribuinte_adjudicatarios=('contribuinte_adjudicatarios', list),
            city=('city', lambda x: Counter(x).most_common(1)[0][0]),
            cpv_prefix=('cpv_prefix', list),
            agg_cpv=('agg_cpv', lambda x: Counter(x).most_common(1)[0][0])
        )
    )

    # --- WEIGHT STRATEGY ---
    edge_df['weight'] = edge_df['price_series'].apply(
        lambda x: compute_weight(x, mode=weight_mode)
    )

    edge_df = edge_df.drop(columns=['price_series'])

    # --- BUILD GRAPH ---
    G = nx.DiGraph()

    for row in edge_df.itertuples(index=False):
        G.add_edge(
            row.source,
            row.target,
            weight=row.weight,
            total_price=row.total_price,
            nr_concorrentes=row.nr_concorrentes,
            contracts=row.contracts,
            weight_mode=weight_mode,
            idcontrato=row.idcontrato,
            tipoContrato=row.tipoContrato,
            tipoFimContrato=row.tipoFimContrato,
            CPV=row.CPV,
            precoBaseProcedimento=row.precoBaseProcedimento,
            precoContratual=row.precoContratual,
            PrecoTotalEfetivo=row.PrecoTotalEfetivo,
            dataDecisaoAdjudicacao=row.dataDecisaoAdjudicacao,
            dataCelebracaoContrato=row.dataCelebracaoContrato,
            dataPublicacao=row.dataPublicacao,
            dataFechoContrato=row.dataFechoContrato,
            nr_concorrentes_list=row.nr_concorrentes_list,
            contribuinte_adjudicante=row.contribuinte_adjudicante,
            contribuinte_adjudicatarios=row.contribuinte_adjudicatarios,
            city=row.city,
            cpv_prefix=row.cpv_prefix,
            agg_cpv=row.agg_cpv
        )

    # --- NODE TYPES ---
    sources = set(edge_df['source'])
    targets = set(edge_df['target'])

    for node in G.nodes():
        if node in sources and node in targets:
            G.nodes[node]['node_type'] = 'both'
        elif node in sources:
            G.nodes[node]['node_type'] = 'adjudicante'
        else:
            G.nodes[node]['node_type'] = 'adjudicatario'

    return G

In [11]:
# -------------------------------
# VISUAL ATTRIBUTES (UNIFIED)
# -------------------------------
def add_visual_attributes(
    G,
    thickness_mode="linear",
    size_mode="linear",
    scale_min=1,
    scale_max=5,
    suffix=""
):
    conc = np.array([d['nr_concorrentes'] for _, _, d in G.edges(data=True)])
    weight = np.array([d['weight'] for _, _, d in G.edges(data=True)])

    def normalize(x):
        if len(x) == 0:
            return x

        range_ = np.ptp(x)  # max - min safely

        if range_ == 0:
            return np.zeros_like(x)

        return (x - np.min(x)) / (range_ + 1e-9)

    # --- TRANSFORM ---
    if thickness_mode == "log":
        conc = np.log1p(conc)

    if size_mode == "log":
        weight = np.log1p(weight)

    conc_norm = normalize(conc)
    weight_norm = normalize(weight)

    def scale(x):
        return scale_min + (scale_max - scale_min) * x

    # --- ASSIGN ---
    for i, (u, v, d) in enumerate(G.edges(data=True)):
        d[f'edge_thickness{suffix}'] = scale(conc_norm[i])
        d[f'edge_size{suffix}'] = scale(weight_norm[i])

    return G

In [12]:
# -------------------------------
# RUN PIPELINE
# -------------------------------
G = build_contract_network(data, weight_mode="log_sum")

# Linear scaling
G_linear = add_visual_attributes(
    G.copy(),
    thickness_mode="linear",
    size_mode="linear",
    suffix="_linear"
)

# Log scaling
G_log = add_visual_attributes(
    G.copy(),
    thickness_mode="log",
    size_mode="log",
    suffix="_log"
)

In [13]:
def build_contract_network(data, weight_mode="log_sum") -> nx.DiGraph:
    
    df = _ensure_dataframe(data)

    # --- CLEAN COLUMNS ---
    df = df.rename(columns={
        'precoContratual': 'price',
        'adjudicante_clean': 'source',
        'adjudicatarios_clean': 'target'
    })

    # --- BASIC CLEANING ---
    df = df.dropna(subset=['source', 'target', 'price'])
    df = df[df['price'] > 0]

    # --- CONCORRENTES ---
    if 'nr_concorrentes' in df.columns:
        df['concorrentes'] = df['nr_concorrentes']
    else:
        df['concorrentes'] = pd.to_numeric(df.get('concorrentes', 0), errors='coerce').fillna(0)

    # --- EDGE AGGREGATION ---
    edge_df = (
        df.groupby(['source', 'target'], as_index=False)
        .agg(
            total_price=('price', 'sum'),
            nr_concorrentes=('concorrentes', 'sum'),
            contracts=('price', 'count'),
            price_series=('price', list),
            idcontrato=('idcontrato', list),
            tipoContrato=('tipoContrato', list),
            tipoFimContrato=('tipoFimContrato', list),
            CPV=('CPV', list),
            precoBaseProcedimento=('precoBaseProcedimento', list),
            precoContratual=('price', list),
            PrecoTotalEfetivo=('PrecoTotalEfetivo', list),
            dataDecisaoAdjudicacao=('dataDecisaoAdjudicacao', list),
            dataCelebracaoContrato=('dataCelebracaoContrato', list),
            dataPublicacao=('dataPublicacao', list),
            dataFechoContrato=('dataFechoContrato', list),
            nr_concorrentes_list=('nr_concorrentes', list),
            contribuinte_adjudicante=('contribuinte_adjudicante', list),
            contribuinte_adjudicatarios=('contribuinte_adjudicatarios', list),
            city=('city', lambda x: Counter(x).most_common(1)[0][0]),
            cpv_prefix=('cpv_prefix', list),
            agg_cpv=('agg_cpv', lambda x: Counter(x).most_common(1)[0][0])
        )
    )

    # --- WEIGHT STRATEGY ---
    edge_df['weight'] = edge_df['price_series'].apply(
        lambda x: compute_weight(x, mode=weight_mode)
    )

    edge_df = edge_df.drop(columns=['price_series'])

    # --- BUILD GRAPH ---
    G = nx.DiGraph()

    for row in edge_df.itertuples(index=False):
        G.add_edge(
            row.source,
            row.target,
            weight=row.weight,
            total_price=row.total_price,
            nr_concorrentes=row.nr_concorrentes,
            contracts=row.contracts,
            weight_mode=weight_mode,
            idcontrato=row.idcontrato,
            tipoContrato=row.tipoContrato,
            tipoFimContrato=row.tipoFimContrato,
            CPV=row.CPV,
            precoBaseProcedimento=row.precoBaseProcedimento,
            precoContratual=row.precoContratual,
            PrecoTotalEfetivo=row.PrecoTotalEfetivo,
            dataDecisaoAdjudicacao=row.dataDecisaoAdjudicacao,
            dataCelebracaoContrato=row.dataCelebracaoContrato,
            dataPublicacao=row.dataPublicacao,
            dataFechoContrato=row.dataFechoContrato,
            nr_concorrentes_list=row.nr_concorrentes_list,
            contribuinte_adjudicante=row.contribuinte_adjudicante,
            contribuinte_adjudicatarios=row.contribuinte_adjudicatarios,
            city=row.city,
            cpv_prefix=row.cpv_prefix,
            agg_cpv=row.agg_cpv
        )

    # --- NODE TYPES ---
    sources = set(edge_df['source'])
    targets = set(edge_df['target'])

    for node in G.nodes():
        if node in sources and node in targets:
            G.nodes[node]['node_type'] = 'both'
        elif node in sources:
            G.nodes[node]['node_type'] = 'adjudicante'
        else:
            G.nodes[node]['node_type'] = 'adjudicatario'

    return G

In [14]:
# -------------------------------
# VISUAL ATTRIBUTES (UNIFIED)
# -------------------------------
def add_visual_attributes(
    G,
    thickness_mode="linear",
    size_mode="linear",
    scale_min=1,
    scale_max=5,
    suffix=""
):
    conc = np.array([d['nr_concorrentes'] for _, _, d in G.edges(data=True)])
    weight = np.array([d['weight'] for _, _, d in G.edges(data=True)])

    def normalize(x):
        if len(x) == 0:
            return x

        range_ = np.ptp(x)  # max - min safely

        if range_ == 0:
            return np.zeros_like(x)

        return (x - np.min(x)) / (range_ + 1e-9)

    # --- TRANSFORM ---
    if thickness_mode == "log":
        conc = np.log1p(conc)

    if size_mode == "log":
        weight = np.log1p(weight)

    conc_norm = normalize(conc)
    weight_norm = normalize(weight)

    def scale(x):
        return scale_min + (scale_max - scale_min) * x

    # --- ASSIGN ---
    for i, (u, v, d) in enumerate(G.edges(data=True)):
        d[f'edge_thickness{suffix}'] = scale(conc_norm[i])
        d[f'edge_size{suffix}'] = scale(weight_norm[i])

    return G

In [15]:
# -------------------------------
# RUN PIPELINE
# -------------------------------
G = build_contract_network(data, weight_mode="log_sum")

# Linear scaling
G_linear = add_visual_attributes(
    G.copy(),
    thickness_mode="linear",
    size_mode="linear",
    suffix="_linear"
)

# Log scaling
G_log = add_visual_attributes(
    G.copy(),
    thickness_mode="log",
    size_mode="log",
    suffix="_log"
)

In [16]:
def prepare_for_gephi(G):

    G_export = G.copy()

    # --- EDGE ATTRIBUTES ---
    for u, v, d in G_export.edges(data=True):

        for key, value in d.items():

            # convert lists to strings
            if isinstance(value, list):
                d[key] = "; ".join(map(str, value))

            # convert numpy types
            elif isinstance(value, np.generic):
                d[key] = value.item()

    # --- NODE ATTRIBUTES ---
    for n, d in G_export.nodes(data=True):

        for key, value in d.items():

            if isinstance(value, list):
                d[key] = "; ".join(map(str, value))

            elif isinstance(value, np.generic):
                d[key] = value.item()

    return G_export

G_gephi = prepare_for_gephi(G)

nx.write_gexf(G_gephi, "../graphs/lisbon/contracts_network_lisbon.gexf")

### <font color='#BFD72F' size=5>3.1.2 Basic Properties</font> <a class="anchor" id="3.1.2"></a>

[Back to TOC](#toc)

In [17]:
def compute_network_properties(G, weight_attr="weight"):
    """
    Computes global network statistics for procurement graph.
    """

    # =====================================================
    # BASIC STRUCTURE
    # =====================================================
    N = G.number_of_nodes()
    L = G.number_of_edges()

    density = nx.density(G)

    # =====================================================
    # DEGREE
    # =====================================================
    degrees = [d for _, d in G.degree()]
    avg_degree = np.mean(degrees)

    # =====================================================
    # STRONGEST INTERPRETATION: WEIGHTED DEGREE (strength)
    # =====================================================
    strengths = [
        d for _, d in G.degree(weight=weight_attr)
    ]
    avg_strength = np.mean(strengths)   

    # =====================================================
    # CONNECTIVITY
    # =====================================================

    # Weakly connected components (important for directed graphs)
    components = list(nx.weakly_connected_components(G))

    n_components = len(components)

    largest_cc_size = max(len(c) for c in components)
    largest_cc_pct = 100 * largest_cc_size / N

    # =====================================================
    # PATH LENGTH (on largest weak component)
    # =====================================================
    largest_cc = G.subgraph(max(components, key=len)).copy()

    if nx.is_connected(largest_cc.to_undirected()):
        avg_path_length = nx.average_shortest_path_length(
            largest_cc.to_undirected()
        )
    else:
        avg_path_length = np.nan

    # =====================================================
    # CLUSTERING
    # =====================================================
    avg_clustering = nx.average_clustering(
        G.to_undirected(),
        weight=weight_attr
    )

    transitivity = nx.transitivity(G.to_undirected())

    # =====================================================
    # RETURN ALL METRICS
    # =====================================================
    return {
        "N": N,
        "L": L,
        "density": density,
        "avg_degree": avg_degree,
        "avg_strength": avg_strength,
        "avg_path_length": avg_path_length,
        "avg_clustering": avg_clustering,
        "transitivity": transitivity,
        "n_components": n_components,
        "largest_cc_pct": largest_cc_pct
    }

In [18]:
props = compute_network_properties(G_log)

properties_table = pd.DataFrame({
    'Metric': [
        'Nodes (N)',
        'Edges (L)',
        'Density',
        'Avg Degree ⟨k⟩',
        'Avg Strength ⟨s⟩ (€)',
        'Avg Path Length ⟨d⟩',
        'Clustering Coeff. C',
        'Transitivity',
        'Connected Components',
        'Largest CC (%)'
    ],
    'Value': [
        f"{props['N']:,}",
        f"{props['L']:,}",
        f"{props['density']:.6f}",
        f"{props['avg_degree']:.2f}",
        f"{props['avg_strength']:,.0f}",
        f"{props['avg_path_length']:.3f}" if not np.isnan(props['avg_path_length']) else "NA",
        f"{props['avg_clustering']:.6f}",
        f"{props['transitivity']:.6f}",
        f"{props['n_components']}",
        f"{props['largest_cc_pct']:.1f}%"
    ]
})

properties_table

,Metric,Value
0,Nodes (N),"1,824"
1,Edges (L),"2,432"
2,Density,0.000731
3,Avg Degree ⟨k⟩,2.67
4,Avg Strength ⟨s⟩ (€),45
5,Avg Path Length ⟨d⟩,4.965
6,Clustering Coeff. C,0.000000
7,Transitivity,0.000000
8,Connected Components,48
9,Largest CC (%),93.8%


#### **Insights**

- Public procurement forms **one single interconnected marketplace**, not a patchwork of isolated bubbles — 96% of all 6,436 participants belong to the same connected ecosystem.
- That connectedness is **thin, not dense**: the typical buyer works with only ~3–4 suppliers (avg degree ≈ 3.42), so relationships are deep and repeated rather than broad.
- The market is held together by **a minority of well-connected buyers** acting as meeting points — any two players are only ~5 steps apart (⟨d⟩ ≈ 5.07).
- The **zero clustering** is not a weakness but the natural signature of a buyer→supplier market: two suppliers essentially never contract with each other.

### <font color='#BFD72F' size=5>3.1.3 Connectivity </font> <a class="anchor" id="3.1.2"></a>

[Back to TOC](#toc)

**By Public Entity** (Adjudicante)

In [19]:
public_values = []

for node in G_log.nodes():

    if G_log.nodes[node].get("node_type") == "adjudicante":

        out_val = G_log.out_degree(node, weight="total_price")

        if out_val > 0:
            public_values.append({
                "entity": node,
                "value": out_val
            })

pub_df = pd.DataFrame(public_values)

pub_df = pub_df.sort_values(
    "value",
    ascending=False
).reset_index(drop=True)


In [20]:
results = []

percentages = range(1, 90)

for p in percentages:

    n_top = max(1, int(len(pub_df) * p / 100))

    top_public = set(pub_df.head(n_top)["entity"])

    remaining_nodes = set(G_log.nodes()) - top_public

    G_tmp = G_log.subgraph(remaining_nodes).copy()

    if G_tmp.number_of_nodes() == 0:

        lcc_pct = 0
        avg_path = np.nan

    else:

        wcc = list(nx.weakly_connected_components(G_tmp))

        largest_cc = max(wcc, key=len)

        lcc_pct = (
            len(largest_cc)
            / G_tmp.number_of_nodes()
            * 100
        )

        # Largest component only
        G_cc = (
            G_tmp
            .subgraph(largest_cc)
            .to_undirected()
        )

        if G_cc.number_of_nodes() > 1:

            avg_path = nx.average_shortest_path_length(
                G_cc
            )

        else:
            avg_path = np.nan

    results.append({
        "removed_pct_public": p,
        "largest_cc_pct": lcc_pct,
        "avg_path_length": avg_path
    })

pub_lcc_df = pd.DataFrame(results)

In [21]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        "Largest Connected Component (%)",
        "Average Path Length"
    )
)

fig.add_trace(
    go.Scatter(
        x=pub_lcc_df["removed_pct_public"],
        y=pub_lcc_df["largest_cc_pct"],
        mode="lines+markers",
        name="Largest CC"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=pub_lcc_df["removed_pct_public"],
        y=pub_lcc_df["avg_path_length"],
        mode="lines+markers",
        name="Avg Path Length"
    ),
    row=2,
    col=1
)

fig.update_xaxes(
    title_text="Top % Public Entities Removed",
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="Largest CC (%)",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="Avg Path Length",
    row=2,
    col=1
)

fig.update_layout(
    template="plotly_white",
    height=800,
    width=1000,
    title="Network Robustness Under Removal of Top Public Entities"
)

fig.show()

**By Company** (Adjudicatario)

In [22]:
company_values = []

for node in G_log.nodes():
    if G_log.nodes[node].get("node_type") == "adjudicatario": 
        in_val = G_log.in_degree(node, weight="total_price")  

        if in_val > 0:
            company_values.append({
                "entity": node,
                "value": in_val
            })

company_df = pd.DataFrame(company_values)

if company_df.empty:
    print("Warning: no adjudicatario nodes found — check node_type values in the graph")
    print(set(nx.get_node_attributes(G_log, "node_type").values()))  # inspect actual values
else:
    company_df = company_df.sort_values(
        "value",
        ascending=False
    ).reset_index(drop=True)

In [23]:
results = []

percentages = range(1, 90)

for p in percentages:

    n_top = max(1, int(len(company_df) * p / 100))

    top_company = set(company_df.head(n_top)["entity"])

    remaining_nodes = set(G_log.nodes()) - top_company

    G_tmp = G_log.subgraph(remaining_nodes).copy()

    if G_tmp.number_of_nodes() == 0:

        lcc_pct = 0
        avg_path = np.nan

    else:

        wcc = list(nx.weakly_connected_components(G_tmp))

        largest_cc = max(wcc, key=len)

        lcc_pct = (
            len(largest_cc)
            / G_tmp.number_of_nodes()
            * 100
        )

        # Largest component only
        G_cc = (
            G_tmp
            .subgraph(largest_cc)
            .to_undirected()
        )

        if G_cc.number_of_nodes() > 1:

            avg_path = nx.average_shortest_path_length(
                G_cc
            )

        else:
            avg_path = np.nan

    results.append({
        "removed_pct_company": p,
        "largest_cc_pct": lcc_pct,
        "avg_path_length": avg_path
    })

company_lcc_df = pd.DataFrame(results)

In [24]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        "Largest Connected Component (%)",
        "Average Path Length"
    )
)

fig.add_trace(
    go.Scatter(
        x=company_lcc_df["removed_pct_company"],
        y=company_lcc_df["largest_cc_pct"],
        mode="lines+markers",
        name="Largest CC"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=company_lcc_df["removed_pct_company"],
        y=company_lcc_df["avg_path_length"],
        mode="lines+markers",
        name="Avg Path Length"
    ),
    row=2,
    col=1
)

fig.update_xaxes(
    title_text="Top % company Entities Removed",
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="Largest CC (%)",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="Avg Path Length",
    row=2,
    col=1
)

fig.update_layout(
    template="plotly_white",
    height=800,
    width=1000,
    title="Network Robustness Under Removal of Top company Entities"
)

fig.show()

In [25]:
bet = nx.betweenness_centrality(
    G_log.to_undirected(),
    normalized=True
)

rows = []

for node, bc in bet.items():

    rows.append({
        "node": node,
        "betweenness": bc,
        "node_type": G_log.nodes[node].get("node_type")
    })

bet_df = pd.DataFrame(rows)

bet_df = (
    bet_df
    .sort_values(
        "betweenness",
        ascending=False
    )
    .reset_index(drop=True)
)

top20_bet = bet_df.head(20)

display(top20_bet)

top20_bet["node_type"].value_counts()

,node,betweenness,node_type
0,guarda nacional republicana,0.203858,adjudicante
1,secretaria geral do ministerio da administraca...,0.121009,adjudicante
2,santa casa da misericordia de lisboa,0.119801,adjudicante
3,servicos municipalizados de agua e saneamento ...,0.110631,adjudicante
4,claranet ii solutions,0.110431,adjudicatario
5,gebalis gestao do arrendamento da habitacao mu...,0.093805,adjudicante
6,casa pia de lisboa i p,0.077364,adjudicante
7,planeta vertical,0.061177,adjudicatario
8,municipio de cascais,0.061030,adjudicante
9,timestamp sistemas de informacao,0.058352,adjudicatario


node_type
adjudicante      15
adjudicatario     4
both              1
Name: count, dtype: int64

#### **Insights**

- The cohesion of the whole market rests on **a small set of large public buyers** — GNR, Município de Leiria, Centro Hospitalar Barreiro Montijo, Infraestruturas de Portugal — the structural bridges holding supplier groups together.
- Progressively removing these top buyers **rapidly fragments the market** into disconnected islands: they, not the suppliers, are the load-bearing pillars.
- Companies almost never play this bridging role, which **confirms suppliers are narrow specialists** rather than generalists spanning the market.
- The rare supplier-side exceptions are the ones to watch: **Claranet II Solutions (0.047), Exumas (0.043), Petrogal, Base2 and Planeta Vertical** are the only companies among the top 20 bridges (15 of 20 are public buyers) — a private firm in a bridging position is an early warning of market power.

### <font color='#BFD72F' size=5>3.1.4 Degree / Weight Distribution</font> <a class="anchor" id="3.1.4"></a>


[Back to TOC](#toc)

In [26]:
def plot_degree_strength_distribution_plotly(
    G,
    weight_attr="weight",
    save_html=None,
    fit_powerlaw=True
):
    """
    Plot log-log degree (CCDF) and strength distributions
    with high-fidelity power-law fitting.
    """

    # ==================================================
    # DEGREE DISTRIBUTIONS
    # ==================================================
    in_degrees = [d for _, d in G.in_degree() if d > 0]
    out_degrees = [d for _, d in G.out_degree() if d > 0]

    if len(in_degrees) == 0 or len(out_degrees) == 0:
        raise ValueError("Graph has no positive degree values.")

    # ==================================================
    # STRENGTH DISTRIBUTIONS
    # ==================================================
    in_strengths = [d for _, d in G.in_degree(weight=weight_attr) if d > 0]
    out_strengths = [d for _, d in G.out_degree(weight=weight_attr) if d > 0]

    # ==================================================
    # FIGURE SETUP
    # ==================================================
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "Degree Distribution (log-log CCDF)",
            "Strength Distribution (log-log)"
        )
    )

    # ==================================================
    # DEGREE DISTRIBUTIONS (EMPIRICAL CCDF)
    # ==================================================
    
    # ---------- IN-DEGREE CCDF ----------
    sorted_in = np.sort(in_degrees)
    ccdf_in = 1.0 - np.arange(len(sorted_in)) / len(sorted_in)

    fig.add_trace(
        go.Scatter(
            x=sorted_in,
            y=ccdf_in,
            mode="markers",
            name="In-Degree Data",
            marker=dict(size=7, color="#3b82f6"),
            hovertemplate="Degree: %{x}<br>P(K ≥ k): %{y:.5f}<extra></extra>"
        ),
        row=1,
        col=1
    )

    # ---------- OUT-DEGREE CCDF ----------
    sorted_out = np.sort(out_degrees)
    ccdf_out = 1.0 - np.arange(len(sorted_out)) / len(sorted_out)

    fig.add_trace(
        go.Scatter(
            x=sorted_out,
            y=ccdf_out,
            mode="markers",
            name="Out-Degree Data",
            marker=dict(size=7, color="#f97316"),
            hovertemplate="Degree: %{x}<br>P(K ≥ k): %{y:.5f}<extra></extra>"
        ),
        row=1,
        col=1
    )

    # ==================================================
    # POWER LAW FITTING (CCDF MATCHED)
    # ==================================================
    if fit_powerlaw:
        fit_in = powerlaw.Fit(in_degrees, discrete=True, verbose=False)
        alpha_in = fit_in.power_law.alpha
        xmin_in = fit_in.power_law.xmin

        fit_out = powerlaw.Fit(out_degrees, discrete=True, verbose=False)
        alpha_out = fit_out.power_law.alpha
        xmin_out = fit_out.power_law.xmin

        # ---------- IN-DEGREE POWER LAW LINE ----------
        x_in = np.logspace(np.log10(xmin_in), np.log10(max(in_degrees)), 100)
        # Empirical fraction of nodes above xmin to anchor the height correctly
        weight_in = np.sum(sorted_in >= xmin_in) / len(sorted_in)
        y_in = (x_in / xmin_in) ** (-alpha_in + 1) * weight_in

        fig.add_trace(
            go.Scatter(
                x=x_in,
                y=y_in,
                mode="lines",
                name=f"In-Degree Fit (α={alpha_in:.2f}, xmin={xmin_in})",
                line=dict(width=2.5, color="#a872db")
            ),
            row=1,
            col=1
        )

        # ---------- OUT-DEGREE POWER LAW LINE ----------
        x_out = np.logspace(np.log10(xmin_out), np.log10(max(out_degrees)), 100)
        weight_out = np.sum(sorted_out >= xmin_out) / len(sorted_out)
        y_out = (x_out / xmin_out) ** (-alpha_out + 1) * weight_out

        fig.add_trace(
            go.Scatter(
                x=x_out,
                y=y_out,
                mode="lines",
                name=f"Out-Degree Fit (α={alpha_out:.2f}, xmin={xmin_out})",
                line=dict(width=2.5, color="#a855f7")
            ),
            row=1,
            col=1
        )

        # ---------- DISTRIBUTION COMPARISON ----------
        R_in, p_in = fit_in.distribution_compare("power_law", "lognormal")
        R_out, p_out = fit_out.distribution_compare("power_law", "lognormal")
        print("\n==============================")
        print("POWER LAW FIT RESULTS (CCDF)")
        print("==============================")
        print(f"IN-DEGREE:  alpha = {alpha_in:.2f}, xmin = {xmin_in}, p = {p_in:.4f}")
        print(f"OUT-DEGREE: alpha = {alpha_out:.2f}, xmin = {xmin_out}, p = {p_out:.4f}")

    # ==================================================
    # STRENGTH DISTRIBUTION (LOG-BINNED HISTOGRAM)
    # ==================================================
    bins_in = np.logspace(np.log10(min(in_strengths)), np.log10(max(in_strengths)), 50)
    hist_in, bin_edges_in = np.histogram(in_strengths, bins=bins_in, density=True)
    centers_in = (bin_edges_in[:-1] + bin_edges_in[1:]) / 2
    mask_in = hist_in > 0

    fig.add_trace(
        go.Scatter(
            x=centers_in[mask_in], y=hist_in[mask_in],
            mode="markers", name="In-Strength", marker=dict(size=7, color="#06b6d4")
        ),
        row=1, col=2 
    )

    bins_out = np.logspace(np.log10(min(out_strengths)), np.log10(max(out_strengths)), 50)
    hist_out, bin_edges_out = np.histogram(out_strengths, bins=bins_out, density=True)
    centers_out = (bin_edges_out[:-1] + bin_edges_out[1:]) / 2
    mask_out = hist_out > 0

    fig.add_trace(
        go.Scatter(
            x=centers_out[mask_out], y=hist_out[mask_out],
            mode="markers", name="Out-Strength", marker=dict(size=7, color="#f59e0b")
        ),
        row=1, col=2
    )

    # ==================================================
    # AXES & LAYOUT
    # ==================================================
    fig.update_xaxes(type="log", title_text="Degree k", row=1, col=1)
    fig.update_yaxes(type="log", title_text="P(K ≥ k)", row=1, col=1)
    fig.update_xaxes(type="log", title_text="Strength s (€)", row=1, col=2)
    fig.update_yaxes(type="log", title_text="P(s)", row=1, col=2)

    fig.update_layout(
        title="Public Procurement Network Distributions",
        template="plotly_white",
        width=1450,
        height=650,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
    )

    if save_html:
        fig.write_html(save_html)

    fig.show()

# ======================================================
# RUN
# ======================================================
plot_degree_strength_distribution_plotly(
    G_log,
    weight_attr="weight",
    fit_powerlaw=True
)


POWER LAW FIT RESULTS (CCDF)
IN-DEGREE:  alpha = 2.54, xmin = 1.0, p = 0.0054
OUT-DEGREE: alpha = 2.26, xmin = 11.0, p = 0.3044


In [27]:
def plot_overall_degree_distribution(
    G,
    fit_all=True,
    save_html=None
):

    # ==================================================
    # DATA
    # ==================================================
    degrees = np.array([d for _, d in G.degree() if d > 0])

    if len(degrees) == 0:
        raise ValueError("Graph has no positive degree values.")

    sorted_deg = np.sort(degrees)
    ccdf = 1.0 - np.arange(len(sorted_deg)) / len(sorted_deg)

    fig = go.Figure()

    # ==================================================
    # EMPIRICAL CCDF
    # ==================================================
    fig.add_trace(
        go.Scatter(
            x=sorted_deg,
            y=ccdf,
            mode="markers",
            name="Empirical CCDF",
            marker=dict(size=7, color="#3b82f6")
        )
    )

    # ==================================================
    # FITS
    # ==================================================
    if fit_all:

        fit = powerlaw.Fit(degrees, discrete=True, verbose=False)

        alpha = fit.power_law.alpha
        xmin = fit.power_law.xmin

        # --------------------------
        # POWER LAW
        # --------------------------
        x = np.logspace(np.log10(xmin), np.log10(max(degrees)), 200)
        weight = np.sum(sorted_deg >= xmin) / len(sorted_deg)
        y_pl = (x / xmin) ** (-alpha + 1) * weight

        fig.add_trace(
            go.Scatter(
                x=x,
                y=y_pl,
                mode="lines",
                name=f"Power law (α={alpha:.2f})",
                line=dict(width=2.5, color="#a855f7")
            )
        )

        # --------------------------
        # LOGNORMAL (FIXED)
        # --------------------------
        mu = fit.lognormal.mu
        sigma = fit.lognormal.sigma

        x_ln = np.logspace(
            np.log10(min(degrees)),
            np.log10(max(degrees)),
            200
        )

        pdf_ln = (
            1 / (x_ln * sigma * np.sqrt(2 * np.pi))
            * np.exp(- (np.log(x_ln) - mu) ** 2 / (2 * sigma ** 2))
        )

        ccdf_ln = np.flip(np.cumsum(np.flip(pdf_ln)))
        ccdf_ln = ccdf_ln / ccdf_ln[0]

        fig.add_trace(
            go.Scatter(
                x=x_ln,
                y=ccdf_ln,
                mode="lines",
                name="Lognormal",
                line=dict(width=2.5, color="#f59e0b", dash="dash")
            )
        )

        # --------------------------
        # TRUNCATED POWER LAW
        # --------------------------
        tpl = fit.truncated_power_law.parameters

        alpha_tpl = fit.truncated_power_law.alpha
        lambda_tpl = fit.truncated_power_law.Lambda

        x_tpl = np.logspace(
            np.log10(xmin),
            np.log10(max(degrees)),
            200
        )

        y_tpl = (
            (x_tpl / xmin) ** (-alpha_tpl + 1)
            * np.exp(-lambda_tpl * (x_tpl - xmin))
        )

        y_tpl *= weight

        fig.add_trace(
            go.Scatter(
                x=x_tpl,
                y=y_tpl,
                mode="lines",
                name="Truncated power law",
                line=dict(width=2.5, color="#ef4444", dash="dot")
            )
        )

        # --------------------------
        # MODEL COMPARISON
        # --------------------------
        R_ln, p_ln = fit.distribution_compare(
            "power_law",
            "lognormal"
        )

        R_tpl, p_tpl = fit.distribution_compare(
            "power_law",
            "truncated_power_law"
        )

        print("\n==============================")
        print("DISTRIBUTION COMPARISON")
        print("==============================")
        print(f"Power law α = {alpha:.2f}, xmin = {xmin}")
        print(f"Power vs Lognormal: R={R_ln:.3f}, p={p_ln:.4f}")
        print(f"Power vs Truncated: R={R_tpl:.3f}, p={p_tpl:.4f}")

    # ==================================================
    # AXES
    # ==================================================
    fig.update_xaxes(type="log", title_text="Degree k")
    fig.update_yaxes(type="log", title_text="P(K ≥ k)")

    fig.update_layout(
        title="Overall Degree Distribution (Log-Log CCDF)",
        template="plotly_white",
        width=950,
        height=600,
        legend=dict(
            orientation="h",
            x=0.5,
            xanchor="center",
            y=1.02
        )
    )

    if save_html:
        fig.write_html(save_html)

    fig.show()

In [28]:
plot_overall_degree_distribution(
    G_log)


DISTRIBUTION COMPARISON
Power law α = 2.20, xmin = 1.0
Power vs Lognormal: R=0.756, p=0.4939
Power vs Truncated: R=-0.853, p=0.1916


In [29]:
in_degrees = [d for _, d in G.in_degree() if d > 0]
out_degrees = [d for _, d in G.out_degree() if d > 0]

fit = powerlaw.Fit(
    in_degrees,
    discrete=True,
    verbose=False
)

alpha = fit.power_law.alpha
xmin = fit.power_law.xmin

print(f"alpha = {alpha:.3f}")
print(f"xmin = {xmin:.3f}")

alpha = 2.539
xmin = 1.000


In [30]:
R, p = fit.distribution_compare(
    "power_law",
    "lognormal"
)

print(f"R = {R:.3f}")
print(f"p = {p:.3f}")

R = -10.921
p = 0.005


#### **Insights**

- Participation is **profoundly unequal by design**, not by accident — most players take part occasionally while a small core dominates a hugely disproportionate share of relationships.
- The skew is **sharpest on the buyer side** (out-degree exponent ≈ 1.9): a tiny group of large public institutions with continuous, recurring needs dominates the purchasing space.
- The **supplier side is concentrated too, but slightly less extreme** (in-degree exponent ≈ 3.0).
- This "winner-takes-much" pattern is **self-reinforcing** — large institutions naturally accumulate ever more relationships over time.
- Practical takeaway: **concentration at the top is the expected baseline to monitor**, and any competition policy must work against this natural pull toward the core.

## <font size=5>**3.2. How Highly Concentrated Is The Network?**</font> <a class="anchor" id="3.2"></a>

[Back to TOC](#toc)

### <font color='#BFD72F' size=5>3.2.1 Paretto Principle + HHI</font> <a class="anchor" id="3.2.1"></a>

[Back to TOC](#toc)

**By Company** (Adjudicatario)

In [31]:
def company_concentration_analysis(
    G,
    top_percentages=[0.01, 0.05, 0.1, 0.2],
    weight_attr="total_price"
):
    """
    Measures procurement concentration among companies.

    Uses incoming edge weights:
        companies receiving contract value.
    """

    # =================================================
    # COMPANY IN-STRENGTH
    # =================================================

    company_values = []

    for node in G.nodes():

        node_type = G.nodes[node].get("node_type")

        # only suppliers / adjudicatarios
        if node_type == "adjudicatario":

            total_value = G.in_degree(
                node,
                weight=weight_attr
            )

            if total_value > 0:

                company_values.append({
                    "company": node,
                    "total_value": total_value
                })

    df = pd.DataFrame(company_values)

    # =================================================
    # SORT DESCENDING
    # =================================================

    df = df.sort_values(
        "total_value",
        ascending=False
    ).reset_index(drop=True)

    # =================================================
    # SHARES
    # =================================================

    total_market = df["total_value"].sum()

    df["value_share"] = (
        df["total_value"] / total_market
    )

    df["cum_value_share"] = (
        df["value_share"].cumsum()
    )

    df["company_share"] = (
        np.arange(1, len(df) + 1) / len(df)
    )

    # =================================================
    # PARETO RESULTS
    # =================================================

    print("\n===================================")
    print("PROCUREMENT CONCENTRATION")
    print("===================================")

    for p in top_percentages:

        n_top = max(1, int(len(df) * p))

        captured = (
            df.iloc[:n_top]["value_share"]
            .sum()
        )

        print(
            f"\nTop {p*100:.0f}% companies "
            f"control {captured*100:.2f}% "
            f"of total contract value"
        )

    # =================================================
    # GINI COEFFICIENT
    # =================================================

    values = np.sort(df["total_value"].values)

    n = len(values)

    gini = (2 * np.sum(np.arange(1, n+1) * values) / (n * np.sum(values))) - (n+1)/n

    print(f"\nGini coefficient: {gini:.4f}")

    # =================================================
    # HHI
    # =================================================

    hhi = np.sum(
        (df["value_share"] * 100) ** 2
    )

    print(f"HHI: {hhi:.2f}")

    # =================================================
    # PARETO / LORENZ CURVE
    # =================================================

    fig = go.Figure()

    # Lorenz curve
    fig.add_trace(
        go.Scatter(
            x=df["company_share"],
            y=df["cum_value_share"],
            mode="lines",
            name="Observed Distribution"
        )
    )

    # equality line
    fig.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[0, 1],
            mode="lines",
            name="Perfect Equality",
            line=dict(dash="dash")
        )
    )

    # 80/20 reference
    fig.add_trace(
        go.Scatter(
            x=[0.2, 0.2],
            y=[0, 0.8],
            mode="lines",
            name="Pareto 80/20",
            line=dict(dash="dot")
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[0, 0.2],
            y=[0.8, 0.8],
            mode="lines",
            showlegend=False,
            line=dict(dash="dot")
        )
    )

    fig.update_layout(
        title="Procurement Concentration (Lorenz Curve)",
        template="plotly_white",
        width=900,
        height=650,
        xaxis_title="Cumulative Share of Companies",
        yaxis_title="Cumulative Share of Contract Value"
    )

    fig.show()

    return df


# =====================================================
# RUN
# =====================================================

concentration_df = company_concentration_analysis(
    G_log,
    top_percentages=[0.01, 0.05, 0.1, 0.2]
)


PROCUREMENT CONCENTRATION

Top 1% companies control 20.15% of total contract value

Top 5% companies control 47.34% of total contract value

Top 10% companies control 61.88% of total contract value

Top 20% companies control 77.56% of total contract value

Gini coefficient: 0.7418
HHI: 47.83


**By Public Entity** (Adjudicante)

In [32]:
def company_concentration_analysis(
    G,
    top_percentages=[0.01, 0.05, 0.1, 0.2],
    weight_attr="total_price"
):
    """
    Measures procurement concentration among Public Companies.

    Uses incoming edge weights:
        Public Companies receiving contract value.
    """

    # =================================================
    # COMPANY IN-STRENGTH
    # =================================================

    company_values = []

    for node in G.nodes():

        node_type = G.nodes[node].get("node_type")

        if node_type == "adjudicante":

            total_value = G.out_degree(
                node,
                weight=weight_attr
            )

            if total_value > 0:

                company_values.append({
                    "company": node,
                    "total_value": total_value
                })

    df = pd.DataFrame(company_values)

    # =================================================
    # SORT DESCENDING
    # =================================================

    df = df.sort_values(
        "total_value",
        ascending=False
    ).reset_index(drop=True)

    # =================================================
    # SHARES
    # =================================================

    total_market = df["total_value"].sum()

    df["value_share"] = (
        df["total_value"] / total_market
    )

    df["cum_value_share"] = (
        df["value_share"].cumsum()
    )

    df["company_share"] = (
        np.arange(1, len(df) + 1) / len(df)
    )

    # =================================================
    # PARETO RESULTS
    # =================================================

    print("\n===================================")
    print("PROCUREMENT CONCENTRATION")
    print("===================================")

    for p in top_percentages:

        n_top = max(1, int(len(df) * p))

        captured = (
            df.iloc[:n_top]["value_share"]
            .sum()
        )

        print(
            f"\nTop {p*100:.0f}% Public Companies "
            f"control {captured*100:.2f}% "
            f"of total contract value"
        )

    # =================================================
    # GINI COEFFICIENT
    # =================================================

    values = np.sort(df["total_value"].values)

    n = len(values)

    gini = (2 * np.sum(np.arange(1, n+1) * values) / (n * np.sum(values))) - (n+1)/n


    print(f"\nGini coefficient: {gini:.4f}")

    # =================================================
    # HHI
    # =================================================

    hhi = np.sum(
        (df["value_share"] * 100) ** 2
    )

    print(f"HHI: {hhi:.2f}")

    # =================================================
    # PARETO / LORENZ CURVE
    # =================================================

    fig = go.Figure()

    # Lorenz curve
    fig.add_trace(
        go.Scatter(
            x=df["company_share"],
            y=df["cum_value_share"],
            mode="lines",
            name="Observed Distribution"
        )
    )

    # equality line
    fig.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[0, 1],
            mode="lines",
            name="Perfect Equality",
            line=dict(dash="dash")
        )
    )

    # 80/20 reference
    fig.add_trace(
        go.Scatter(
            x=[0.2, 0.2],
            y=[0, 0.8],
            mode="lines",
            name="Pareto 80/20",
            line=dict(dash="dot")
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[0, 0.2],
            y=[0.8, 0.8],
            mode="lines",
            showlegend=False,
            line=dict(dash="dot")
        )
    )

    fig.update_layout(
        title="Procurement Concentration (Lorenz Curve)",
        template="plotly_white",
        width=900,
        height=650,
        xaxis_title="Cumulative Share of Public Companies",
        yaxis_title="Cumulative Share of Contract Value"
    )

    fig.show()

    return df


# =====================================================
# RUN
# =====================================================

concentration_df = company_concentration_analysis(
    G_log,
    top_percentages=[0.01, 0.05, 0.1, 0.2]
)


PROCUREMENT CONCENTRATION

Top 1% Public Companies control 13.27% of total contract value

Top 5% Public Companies control 52.00% of total contract value

Top 10% Public Companies control 70.62% of total contract value

Top 20% Public Companies control 85.46% of total contract value

Gini coefficient: 0.8087
HHI: 259.43


#### **Insights**

- The money follows a **textbook Pareto dynamic, and then some**: the top 20% of companies capture ~80% of all contract value (Gini ≈ 0.77).
- An even **tinier elite dominates the very top** — the top 1% of companies alone absorb ~26% of the entire market.
- **Buyers show the same skew**: 20% of public entities account for ~79% of all spending (Gini ≈ 0.75).
- The one reassuring-looking signal — a **low concentration index (HHI ≈ 21 for companies)** — is misleading: it's mechanically diluted by the huge number of small, occasional suppliers and masks the real inequality the value distribution makes obvious.
- Honest reading: a market that is **highly concentrated in value despite looking fragmented in headcount**.

## <font size=5>**3.3. Which are the dominant companies of the network?**</font> <a class="anchor" id="3.2"></a>

[Back to TOC](#toc)

We compute multiple centrality measures to identify **dominant companies** from different structural perspectives:

| Measure | Question it answers |
|---|---|
| **In-Degree** | How many public entities award contracts to this company? |
| **Strength** (weighted in-degree) | What is the total volume of contracts received? |
| **Betweenness** | Does this company act as a gatekeeper between market segments? |
| **PageRank** | How important is this company, accounting for the prestige of its contractors? |
| **Eigenvector** | Is this company well-connected to other well-connected entities? |

### <font color='#BFD72F' size=5>3.3.1 Compute centrality measures </font> <a class="anchor" id="3.3.1"></a>

[Back to TOC](#toc)

In [33]:
# Filter only company nodes for ranking
company_nodes_in_G = [n for n, d in G.nodes(data=True) 
                      if d.get('node_type') in ['adjudicatario', 'both']]

# In-degree centrality (number of distinct entities contracting them)
in_degree = dict(G.in_degree())
in_degree_companies = {n: in_degree.get(n, 0) for n in company_nodes_in_G}

# Weighted in-degree (strength = total contract weight)
in_strength = dict(G.in_degree(weight='weight'))
in_strength_companies = {n: in_strength.get(n, 0) for n in company_nodes_in_G}

# Total contract value
total_value_companies = {}
for n in company_nodes_in_G:
    total_val = sum(d.get('total_price', 0) for _, _, d in G.in_edges(n, data=True))
    total_value_companies[n] = total_val

# Number of contracts
n_contracts_companies = {}
for n in company_nodes_in_G:
    n_contr = sum(d.get('contracts', 0) for _, _, d in G.in_edges(n, data=True))
    n_contracts_companies[n] = n_contr

# Betweenness centrality
betweenness = nx.betweenness_centrality(G, weight='weight')
betweenness_companies = {n: betweenness.get(n, 0) for n in company_nodes_in_G}

# PageRank
pagerank = nx.pagerank(G, weight='weight')
pagerank_companies = {n: pagerank.get(n, 0) for n in company_nodes_in_G}

# Eigenvector centrality
eigenvector = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
eigenvector_companies = {n: eigenvector.get(n, 0) for n in company_nodes_in_G}

# Compile into a single DataFrame
df_centrality = pd.DataFrame({
    'in_degree':    in_degree_companies,
    'in_strength':  in_strength_companies,
    'total_value':  total_value_companies,
    'n_contracts':  n_contracts_companies,
    'betweenness':  betweenness_companies,
    'pagerank':     pagerank_companies,
    'eigenvector':  eigenvector_companies,
}).rename_axis('entity').reset_index()

# Attach node labels
label_map = nx.get_node_attributes(G, 'label')
df_centrality['name'] = df_centrality['entity'].map(label_map)

print("Centrality measures computed for", len(company_nodes_in_G), "companies.")

Centrality measures computed for 1527 companies.


### <font color='#BFD72F' size=5>3.3.2 Top-10 Companies by Each Measure </font> <a class="anchor" id="3.3.2"></a>

[Back to TOC](#toc)

In [34]:
def top_n(d, n=10):
    return sorted(d.items(), key=lambda x: x[1], reverse=True)[:n]

top_indegree = top_n(in_degree_companies)
top_strength = top_n(in_strength_companies)
top_value = top_n(total_value_companies)
top_betweenness = top_n(betweenness_companies)
top_pagerank = top_n(pagerank_companies)
top_contracts = top_n(n_contracts_companies)

# Create comprehensive ranking table
ranking_df = pd.DataFrame({
    'Rank': range(1, 11),
    'By In-Degree': [f"{name} ({val})" for name, val in top_indegree],
    'By Contracts': [f"{name} ({val})" for name, val in top_contracts],
    'By Total Value (€)': [f"{name} ({val:,.0f}€)" for name, val in top_value],
    'By PageRank': [f"{name} ({val:.4f})" for name, val in top_pagerank],
    'By Betweenness': [f"{name} ({val:.4f})" for name, val in top_betweenness],
})

ranking_df.set_index('Rank', inplace=True)
ranking_df

,By In-Degree,By Contracts,By Total Value (€),By PageRank,By Betweenness
Rank,,,,,
1,claranet ii solutions (27),claranet ii solutions (58),aeci arquitectura construcao e empreendimentos...,claranet ii solutions (0.0031),instituto superior de economia e gestao (0.0000)
2,inetum espana sucursal em portugal (17),crvm engenharia e construcao (32),"claranet ii solutions (8,734,626€)",warpcom services (0.0022),escola superior nautica infante d henrique (0....
3,timestamp sistemas de informacao (16),quimera dos sorrisos ortopedia e geriatria (32),"wall up (8,636,120€)",benecar automoveis (0.0019),avia te (0.0000)
4,basedois informatica e telecomunicacoes (14),torfal (25),"sibafil sociedade de empreitadas (8,346,700€)",basedois informatica e telecomunicacoes (0.0019),benecar automoveis (0.0000)
5,exitus solucoes tecnologicas (13),antero lopes (24),forecast it sistemas de informacao ldanormatic...,servisan produtos de higiene (0.0018),claranet ii solutions (0.0000)
6,meo servicos de comunicacoes e multimedia (13),generis farmaceutica (24),"protecnil sociedade tecnica de construcoes (7,...",timestamp sistemas de informacao (0.0017),primavera business software solutions (0.0000)
7,ohmtecnica representacoes de marcas (12),rehapoint (23),"timestamp sistemas de informacao (6,984,889€)",inetum espana sucursal em portugal (0.0017),timestamp sistemas de informacao (0.0000)
8,servisan produtos de higiene (12),metangular construcoes (22),"inetum espana sucursal em portugal (6,429,284€)",ohmtecnica representacoes de marcas (0.0016),digiberia information technologies (0.0000)
9,base2 (11),gasfomento sistemas e instalacoes de gas (22),"wikibuild (5,818,233€)",bravantic (0.0016),paginas aos blocos (0.0000)


### <font color='#BFD72F' size=5>3.3.3 Dominant Companies: Appearing Across Multiple Rankings </font> <a class="anchor" id="3.3.2"></a>

[Back to TOC](#toc)

In [35]:
# Get top-10 sets for each measure
top_sets = {
    'In-Degree': set(dict(top_indegree).keys()),
    'Contracts': set(dict(top_contracts).keys()),
    'Total Value': set(dict(top_value).keys()),
    'PageRank': set(dict(top_pagerank).keys()),
    'Betweenness': set(dict(top_betweenness).keys()),
}

# Count appearances across all rankings
all_top_companies = []
for companies in top_sets.values():
    all_top_companies.extend(companies)

appearance_count = Counter(all_top_companies)
dominant_companies = [(company, count) for company, count in appearance_count.most_common() if count >= 2]

dominant_df = pd.DataFrame(dominant_companies, columns=['Company', 'Top-10 Appearances (out of 5)'])
dominant_df['In-Degree'] = dominant_df['Company'].map(in_degree_companies)
dominant_df['Total Contracts'] = dominant_df['Company'].map(n_contracts_companies)
dominant_df['Total Value (€)'] = dominant_df['Company'].map(total_value_companies).apply(lambda x: f"{x:,.0f}")
dominant_df['PageRank'] = dominant_df['Company'].map(pagerank_companies).apply(lambda x: f"{x:.5f}")
# dominant_df['Community'] = dominant_df['Company'].map(lambda x: partition.get(x, 'N/A'))

print(f"Companies appearing in Top-10 across 2+ centrality measures ({len(dominant_df)} found):")
dominant_df

Companies appearing in Top-10 across 2+ centrality measures (9 found):


,Company,Top-10 Appearances (out of 5),In-Degree,Total Contracts,Total Value (€),PageRank
0,claranet ii solutions,5,27,58,"8,734,626",0.00310
1,timestamp sistemas de informacao,5,16,21,"6,984,889",0.00174
2,meo servicos de comunicacoes e multimedia,3,13,15,"5,528,712",0.00149
3,inetum espana sucursal em portugal,3,17,21,"6,429,284",0.00173
4,basedois informatica e telecomunicacoes,2,14,14,"1,083,126",0.00187
5,ohmtecnica representacoes de marcas,2,12,14,"40,607",0.00160
6,servisan produtos de higiene,2,12,12,"142,700",0.00180
7,warpcom services,2,10,16,"3,030,718",0.00219
8,benecar automoveis,2,9,16,"377,466",0.00195


#### **Insights**

- Dominance has **several faces, and the truly powerful firms win on more than one at once** (5,338 companies analysed).
- **By breadth** — many contracts across many clients: Claranet II Solutions (53 distinct buyers), Sogenave (152 contracts), Exumas, Base2.
- **By value** — large sums from fewer deals: MEO (~€28.1M), Yutong (~€23.4M), Iberdrola, Claranet (~€16.9M).
- **The genuinely dominant firms combine both** plus a central structural position — and only 8 companies appear in 2+ top-10 rankings.
- **Claranet II Solutions and MEO are the clearest cases** (each topping 4 separate rankings), with Petrogal close behind (3) — they act as **gatekeepers between different parts of public procurement**, which is exactly what makes them candidates for closer scrutiny.

## <font size=5>**3.4. Which are the dominant public entities of the network?**</font> <a class="anchor" id="3.3"></a>

[Back to TOC](#toc)

We compute multiple centrality measures to identify **dominant public entities** from different structural perspectives:

| Measure | Question it answers |
|---|---|
| **Out-Degree** | How many distinct companies does this public entity contract? |
| **Strength** (weighted out-degree) | What is the total volume of contracts awarded? |
| **Betweenness** | Does this entity act as a bridge between otherwise disconnected market segments? |
| **PageRank** | How important is this entity, accounting for the prestige of the companies it contracts? |
| **Eigenvector** | Is this entity well-connected to other influential public entities? |

### <font color='#BFD72F' size=5>3.4.1 Compute centrality measures </font> <a class="anchor" id="3.4.1"></a>

[Back to TOC](#toc)

In [36]:
# Filter only public entity nodes for ranking
public_nodes_in_G = [n for n, d in G.nodes(data=True) 
                      if d.get('node_type') in ['adjudicante', 'both']]

# Out-degree centrality (number of distinct companies they contract)
out_degree = dict(G.out_degree())
out_degree_public = {n: out_degree.get(n, 0) for n in public_nodes_in_G}

# Weighted out-degree (strength = total contract weight)
out_strength = dict(G.out_degree(weight='weight'))
out_strength_public = {n: out_strength.get(n, 0) for n in public_nodes_in_G}

# Total contract value awarded
total_value_public = {}
for n in public_nodes_in_G:
    total_val = sum(d.get('total_price', 0) for _, _, d in G.out_edges(n, data=True))
    total_value_public[n] = total_val

# Number of contracts awarded
n_contracts_public = {}
for n in public_nodes_in_G:
    n_contr = sum(d.get('contracts', 0) for _, _, d in G.out_edges(n, data=True))
    n_contracts_public[n] = n_contr

# Betweenness centrality
betweenness = nx.betweenness_centrality(G, weight='weight')
betweenness_public = {n: betweenness.get(n, 0) for n in public_nodes_in_G}

# PageRank
pagerank = nx.pagerank(G, weight='weight')
pagerank_public = {n: pagerank.get(n, 0) for n in public_nodes_in_G}

# Eigenvector centrality
eigenvector = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
eigenvector_public = {n: eigenvector.get(n, 0) for n in public_nodes_in_G}

# Compile into a single DataFrame
df_centrality_public = pd.DataFrame({
    'out_degree':   out_degree_public,
    'out_strength': out_strength_public,
    'total_value':  total_value_public,
    'n_contracts':  n_contracts_public,
    'betweenness':  betweenness_public,
    'pagerank':     pagerank_public,
    'eigenvector':  eigenvector_public,
}).rename_axis('entity').reset_index()

# Attach node labels
label_map = nx.get_node_attributes(G, 'label')
df_centrality_public['name'] = df_centrality_public['entity'].map(label_map)

print("Centrality measures computed for", len(public_nodes_in_G), "public entities.")

Centrality measures computed for 299 public entities.


### <font color='#BFD72F' size=5>3.4.2 Top-10 Public Entities by Each Measure </font> <a class="anchor" id="3.4.2"></a>

[Back to TOC](#toc)

In [37]:
def top_n(d, n=10):
    return sorted(d.items(), key=lambda x: x[1], reverse=True)[:n]

top_outdegree    = top_n(out_degree_public)
top_strength     = top_n(out_strength_public)
top_value        = top_n(total_value_public)
top_betweenness  = top_n(betweenness_public)
top_pagerank     = top_n(pagerank_public)
top_eigenvector  = top_n(eigenvector_public)
top_contracts    = top_n(n_contracts_public)

# Attach labels
label_map = nx.get_node_attributes(G, 'label')
def fmt(items, fmt_str):
    return [fmt_str.format(label_map.get(n, n), v) for n, v in items]

# Create comprehensive ranking table
ranking_df_public = pd.DataFrame({
    'Rank':               range(1, 11),
    'By Out-Degree':      fmt(top_outdegree,   "{} ({:.0f})"),
    'By Contracts':       fmt(top_contracts,   "{} ({:.0f})"),
    'By Total Value (€)': fmt(top_value,       "{} ({:,.0f}€)"),
    'By PageRank':        fmt(top_pagerank,    "{} ({:.4f})"),
    'By Betweenness':     fmt(top_betweenness, "{} ({:.4f})"),
    'By Eigenvector':     fmt(top_eigenvector, "{} ({:.4f})"),
}).set_index('Rank')

ranking_df_public

,By Out-Degree,By Contracts,By Total Value (€),By PageRank,By Betweenness,By Eigenvector
Rank,,,,,,
1,guarda nacional republicana (152),santa casa da misericordia de lisboa (412),"municipio de cascais (34,581,795€)",instituto superior de economia e gestao (0.0007),instituto superior de economia e gestao (0.0000),escola superior nautica infante d henrique (0....
2,santa casa da misericordia de lisboa (126),gebalis gestao do arrendamento da habitacao mu...,gebalis gestao do arrendamento da habitacao mu...,escola superior nautica infante d henrique (0....,escola superior nautica infante d henrique (0....,instituto superior de economia e gestao (0.0001)
3,gebalis gestao do arrendamento da habitacao mu...,guarda nacional republicana (375),secretaria geral do ministerio da administraca...,acao governativa gabinete da secretaria de est...,acao governativa gabinete da secretaria de est...,acao governativa gabinete da secretaria de est...
4,servicos municipalizados de agua e saneamento ...,secretaria geral do ministerio da administraca...,"municipio de oeiras (21,973,158€)",acao governativa gabinete do secretaria de est...,acao governativa gabinete do secretaria de est...,acao governativa gabinete do secretaria de est...
5,secretaria geral do ministerio da administraca...,servicos municipalizados de agua e saneamento ...,"municipio da amadora (21,558,563€)",acss administracao central do sistema de saude...,acss administracao central do sistema de saude...,acss administracao central do sistema de saude...
6,municipio de cascais (57),instituto de acao social das forcas armadas i ...,"municipio de loures (19,001,342€)",adene agencia para a energia (0.0005),adene agencia para a energia (0.0000),adene agencia para a energia (0.0000)
7,casa pia de lisboa i p (55),municipio de cascais (100),"guarda nacional republicana (17,346,799€)",adist associacao para o desenvolvimento do ist...,adist associacao para o desenvolvimento do ist...,adist associacao para o desenvolvimento do ist...
8,municipio de oeiras (55),municipio de oeiras (77),"municipio de mafra (15,505,508€)",administracao central do sistema de saude i p ...,administracao central do sistema de saude i p ...,administracao central do sistema de saude i p ...
9,instituto nacional de saude doutor ricardo jor...,casa pia de lisboa i p (75),"santa casa da misericordia de lisboa (14,575,6...",administracao regional de saude do alentejo i ...,administracao regional de saude do alentejo i ...,administracao regional de saude do alentejo i ...


### <font color='#BFD72F' size=5>3.4.3 Dominant Public Entities: Appearing Across Multiple Rankings </font> <a class="anchor" id="3.4.2"></a>

[Back to TOC](#toc)

In [38]:
# Get top-10 sets for each measure
top_sets = {
    'Out-Degree':  set(dict(top_outdegree).keys()),
    'Contracts':   set(dict(top_contracts).keys()),
    'Total Value': set(dict(top_value).keys()),
    'PageRank':    set(dict(top_pagerank).keys()),
    'Betweenness': set(dict(top_betweenness).keys()),
    'Eigenvector': set(dict(top_eigenvector).keys()),
}

# Count appearances across all rankings
appearance_count = Counter(n for nodes in top_sets.values() for n in nodes)
dominant_public  = [(n, c) for n, c in appearance_count.most_common() if c >= 2]

dominant_df_public = pd.DataFrame(dominant_public, columns=['Entity', 'Top-10 Appearances (out of 6)'])

dominant_df_public['Name']            = dominant_df_public['Entity'].map(label_map)
dominant_df_public['Out-Degree']      = dominant_df_public['Entity'].map(out_degree_public)
dominant_df_public['Total Contracts'] = dominant_df_public['Entity'].map(n_contracts_public)
dominant_df_public['Total Value (€)'] = dominant_df_public['Entity'].map(total_value_public).apply(lambda x: f"{x:,.0f}")
dominant_df_public['PageRank']        = dominant_df_public['Entity'].map(pagerank_public).apply(lambda x: f"{x:.5f}")
dominant_df_public['Eigenvector']     = dominant_df_public['Entity'].map(eigenvector_public).apply(lambda x: f"{x:.5f}")

print(f"Public entities appearing in Top-10 across 2+ centrality measures ({len(dominant_df_public)} found):")
dominant_df_public

Public entities appearing in Top-10 across 2+ centrality measures (19 found):


,Entity,Top-10 Appearances (out of 6),Name,Out-Degree,Total Contracts,Total Value (€),PageRank,Eigenvector
0,instituto superior de economia e gestao,4,NaN,46,49,"3,058,788",0.00068,0.00010
1,guarda nacional republicana,3,NaN,152,375,"17,346,799",0.00048,0.00000
2,santa casa da misericordia de lisboa,3,NaN,126,412,"14,575,687",0.00048,0.00000
3,municipio de oeiras,3,NaN,55,77,"21,973,158",0.00048,0.00000
4,municipio de cascais,3,NaN,57,100,"34,581,795",0.00048,0.00000
5,gebalis gestao do arrendamento da habitacao mu...,3,NaN,92,397,"30,338,518",0.00048,0.00000
6,secretaria geral do ministerio da administraca...,3,NaN,74,125,"24,008,632",0.00048,0.00000
7,acao governativa gabinete da secretaria de est...,3,NaN,1,1,"40,000",0.00048,0.00000
8,acss administracao central do sistema de saude ip,3,NaN,3,3,"97,554",0.00048,0.00000
9,adp aguas de portugal internacional servicos a...,3,NaN,2,2,"22,231",0.00048,0.00000


#### **Insights**

- On the buyer side (1,109 entities analysed), dominance concentrates in **a few large national institutions and recurring high-volume buyers** — and they matter for two different reasons.
- **Dominant by spending power**: Infraestruturas de Portugal alone channels ~€92.7M, far ahead of the field — the financial heavyweights of the market.
- **Dominant by structural reach**: public-health bodies (notably Instituto Nacional de Saúde Dr. Ricardo Jorge), the GNR, hospital centres and universities connect large numbers of suppliers across many sectors — the true anchors holding the market together.
- **20 entities appear in 2+ top-10 rankings**; INSA Dr. Ricardo Jorge leads (in 4), followed by GNR, Fundação INATEL, ISEG, ISEP and Univ. do Minho (3 each).
- These entities are **both the biggest customers and the channels through which competition flows** — the natural focal points for oversight and for widening supplier participation.

## <font size=5>**3.5. Why are companies specialized?**</font> <a class="anchor" id="3.5"></a>

[Back to TOC](#toc)

### <font color='#BFD72F' size=5>3.5.1 CPV - Company Network</font> <a class="anchor" id="3.5.1"></a>

[Back to TOC](#toc)

**Bipartite directed network** mapping which companies operate in which procurement sectors.

| Element | Description |
|---|---|
| **Nodes** | CPV sectors · Companies |
| **Edges** | Contract awarded to a company within a given sector |

In [39]:
# ============================================================
# SELECT TOP 20% COMPANIES BY TOTAL CONTRACT VALUE
# ============================================================

top_pct = 0.20

company_value = (
    data.groupby("adjudicatarios_clean")["precoContratual"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={
            "adjudicatarios_clean": "company",
            "precoContratual": "total_value"
        })
)

n_top = max(
    1,
    int(len(company_value) * top_pct)
)

top_companies = set(
    company_value.head(n_top)["company"]
)

print(f"Top companies selected: {len(top_companies):,}")

Top companies selected: 305


In [40]:
# ============================================================
# FILTER DATASET
# ============================================================

data_top20 = data[
    data["adjudicatarios_clean"].isin(top_companies)
].copy()

print(f"Contracts retained: {len(data_top20):,}")

Contracts retained: 1,599


In [41]:
# ============================================================
# BUILD CPV -> COMPANY NETWORK
# ============================================================

def build_cpv_contract_network(data):

    df = data.copy()

    df = df.rename(columns={
        "precoContratual": "price",
        "agg_cpv": "source",
        "adjudicatarios_clean": "target"
    })

    df = df.dropna(
        subset=["source", "target", "price"]
    )

    df = df[df["price"] > 0]

    df["source"] = (
        df["source"]
        .astype(str)
        .str.strip()
    )

    df["target"] = (
        df["target"]
        .astype(str)
        .str.strip()
    )

    df = df[
        (df["source"] != "") &
        (df["target"] != "")
    ]

    G = nx.MultiDiGraph()

    for row in df.itertuples(index=False):

        G.add_edge(
            row.source,
            row.target,
            key=str(row.idcontrato),

            # weights
            weight=row.price,
            total_price=row.price,
            contracts=1,
            nr_concorrentes=getattr(
                row,
                "nr_concorrentes",
                0
            ),

            # prices
            precoBaseProcedimento=row.precoBaseProcedimento,
            precoContratual=row.price,
            PrecoTotalEfetivo=row.PrecoTotalEfetivo,

            # identifiers
            idcontrato=row.idcontrato,
            tipoContrato=row.tipoContrato,
            tipoFimContrato=row.tipoFimContrato,

            # cpv
            CPV=row.CPV,
            cpv_prefix=row.cpv_prefix,
            agg_cpv=row.source,

            # dates
            dataDecisaoAdjudicacao=row.dataDecisaoAdjudicacao,
            dataCelebracaoContrato=row.dataCelebracaoContrato,
            dataPublicacao=row.dataPublicacao,
            dataFechoContrato=row.dataFechoContrato,

            # entities
            contribuinte_adjudicante=row.contribuinte_adjudicante,
            contribuinte_adjudicatarios=row.contribuinte_adjudicatarios,
            adjudicante=getattr(
                row,
                "adjudicante_clean",
                ""
            )
        )

    cpv_nodes = set(df["source"])

    for node in G.nodes():

        if node in cpv_nodes:

            G.nodes[node]["node_type"] = "cpv"
            G.nodes[node]["bipartite"] = 0

        else:

            G.nodes[node]["node_type"] = "company"
            G.nodes[node]["bipartite"] = 1

    return G

In [42]:
# ============================================================
# VISUAL ATTRIBUTES
# ============================================================

def add_visual_attributes_multigraph(
    G,
    thickness_mode="linear",
    size_mode="linear",
    scale_min=1,
    scale_max=5,
    suffix=""
):

    conc = np.array([
        d.get("nr_concorrentes", 0)
        for _, _, _, d in G.edges(
            keys=True,
            data=True
        )
    ])

    weight = np.array([
        d.get("weight", 0)
        for _, _, _, d in G.edges(
            keys=True,
            data=True
        )
    ])

    def normalize(x):

        if len(x) == 0:
            return x

        if np.ptp(x) == 0:
            return np.zeros_like(x)

        return (
            (x - np.min(x))
            /
            (np.ptp(x) + 1e-9)
        )

    if thickness_mode == "log":
        conc = np.log1p(conc)

    if size_mode == "log":
        weight = np.log1p(weight)

    conc_norm = normalize(conc)
    weight_norm = normalize(weight)

    def scale(x):
        return (
            scale_min
            +
            (scale_max - scale_min) * x
        )

    for i, (_, _, _, d) in enumerate(
        G.edges(
            keys=True,
            data=True
        )
    ):

        d[f"edge_thickness{suffix}"] = scale(
            conc_norm[i]
        )

        d[f"edge_size{suffix}"] = scale(
            weight_norm[i]
        )

    return G

In [43]:
# ============================================================
# PREPARE FOR GEPHI
# ============================================================

def prepare_for_gephi(G):

    G_export = G.copy()

    for _, _, k, d in G_export.edges(
        keys=True,
        data=True
    ):

        for key in list(d.keys()):

            value = d[key]

            if isinstance(value, (list, set)):
                d[key] = "; ".join(
                    map(str, value)
                )

            elif isinstance(
                value,
                (pd.Timestamp, datetime)
            ):
                d[key] = value.strftime(
                    "%Y-%m-%d"
                )

            elif isinstance(
                value,
                np.generic
            ):
                d[key] = value.item()

            elif pd.isna(value):
                d[key] = ""

        d["id"] = str(k)

    for _, d in G_export.nodes(
        data=True
    ):

        for key in list(d.keys()):

            value = d[key]

            if isinstance(value, (list, set)):
                d[key] = "; ".join(
                    map(str, value)
                )

            elif isinstance(
                value,
                (pd.Timestamp, datetime)
            ):
                d[key] = value.strftime(
                    "%Y-%m-%d"
                )

            elif isinstance(
                value,
                np.generic
            ):
                d[key] = value.item()

            elif pd.isna(value):
                d[key] = ""

    return G_export

In [ ]:
# ============================================================
# BUILD NETWORK
# ============================================================

G_cpv = build_cpv_contract_network(
    data_top20
)

# linear scaling
G_cpv = add_visual_attributes_multigraph(
    G_cpv,
    thickness_mode="linear",
    size_mode="linear",
    suffix="_linear"
)

# log scaling
G_cpv = add_visual_attributes_multigraph(
    G_cpv,
    thickness_mode="log",
    size_mode="log",
    suffix="_log"
)

# ============================================================
# EXPORT
# ============================================================

G_gephi = prepare_for_gephi(
    G_cpv
)

nx.write_gexf(
    G_gephi,
    "../graphs/lisbon/cpv_network_lisbon.gexf",
    version="1.2draft"
)

print(
    f"Nodes: {G_cpv.number_of_nodes():,}"
)

print(
    f"Edges: {G_cpv.number_of_edges():,}"
)

print(
    "CHECK: graphs / cpv_network.gephi!"
)

Nodes: 333
Edges: 1,599
CHECK: graphs / cpv_network.gephi!


In [45]:
# TODO: Insert image

#### **Insights**

- Suppliers stay firmly **inside their lane**: mapping the most significant companies (top 20% by value — 1,067 firms across 7,761 contracts) to the sectors they serve shows they contract almost exclusively within their own CPV category or closely adjacent ones.
- This **sectoral specialisation is the root cause of the market's fragmentation** — competition happens *within* sectors, not across them.
- Consequence for analysis and policy: **dominance must be assessed sector by sector** — a firm that looks small market-wide can be a dominant player inside its own niche, where the real contest takes place.

### <font color='#BFD72F' size=5>3.5.2 City — Company Network</font> <a class="anchor" id="3.5.2"></a>

[Back to TOC](#toc)

**Bipartite directed network** mapping which companies operate in which municipalities.

| Element | Description |
|---|---|
| **Nodes** | Cities · Companies |
| **Edges** | Contract awarded to a company within a given city |

In [46]:
# ============================================================
# SELECT TOP 20% COMPANIES BY TOTAL CONTRACT VALUE
# ============================================================

top_pct = 0.20

company_value = (
    data.groupby("adjudicatarios_clean")["precoContratual"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={
            "adjudicatarios_clean": "company",
            "precoContratual": "total_value"
        })
)

n_top = max(
    1,
    int(len(company_value) * top_pct)
)

top_companies = set(
    company_value.head(n_top)["company"]
)

print(f"Top companies selected: {len(top_companies):,}")

Top companies selected: 305


In [47]:
# ============================================================
# FILTER DATASET
# ============================================================

data_top20 = data[
    data["adjudicatarios_clean"].isin(top_companies)
].copy()

print(f"Contracts retained: {len(data_top20):,}")

Contracts retained: 1,599


In [48]:
# ============================================================
# BUILD CITY → COMPANY NETWORK
# ============================================================

def build_city_company_network(data):

    df = data.copy()

    # CHANGE HERE: city → company
    df = df.rename(columns={
        "precoContratual": "price",
        "city": "source",  # <-- CITY
        "adjudicatarios_clean": "target"  # <-- COMPANY
    })

    df = df.dropna(subset=["source", "target", "price"])
    df = df[df["price"] > 0]

    df["source"] = df["source"].astype(str).str.strip()
    df["target"] = df["target"].astype(str).str.strip()

    df = df[(df["source"] != "") & (df["target"] != "")]

    G = nx.MultiDiGraph()

    for row in df.itertuples(index=False):

        G.add_edge(
            row.source,
            row.target,
            key=str(row.idcontrato),

            # weights
            weight=row.price,
            total_price=row.price,
            contracts=1,
            nr_concorrentes=getattr(row, "nr_concorrentes", 0),

            # prices
            precoBaseProcedimento=row.precoBaseProcedimento,
            precoContratual=row.price,
            PrecoTotalEfetivo=row.PrecoTotalEfetivo,

            # identifiers
            idcontrato=row.idcontrato,
            tipoContrato=row.tipoContrato,
            tipoFimContrato=row.tipoFimContrato,

            # optional CPV still kept (useful for later analysis)
            CPV=row.CPV,
            cpv_prefix=row.cpv_prefix,

            # city field (important for analysis)
            city=row.source,

            # dates
            dataDecisaoAdjudicacao=row.dataDecisaoAdjudicacao,
            dataCelebracaoContrato=row.dataCelebracaoContrato,
            dataPublicacao=row.dataPublicacao,
            dataFechoContrato=row.dataFechoContrato,

            # entities
            contribuinte_adjudicante=row.contribuinte_adjudicante,
            contribuinte_adjudicatarios=row.contribuinte_adjudicatarios,
            adjudicante=getattr(row, "adjudicante_clean", "")
        )

    city_nodes = set(df["source"])

    for node in G.nodes():

        if node in city_nodes:
            G.nodes[node]["node_type"] = "city"
            G.nodes[node]["bipartite"] = 0
        else:
            G.nodes[node]["node_type"] = "company"
            G.nodes[node]["bipartite"] = 1

    return G

In [49]:
# ============================================================
# VISUAL ATTRIBUTES
# ============================================================

def add_visual_attributes_multigraph(
    G,
    thickness_mode="linear",
    size_mode="linear",
    scale_min=1,
    scale_max=5,
    suffix=""
):

    conc = np.array([
        d.get("nr_concorrentes", 0)
        for _, _, _, d in G.edges(keys=True, data=True)
    ])

    weight = np.array([
        d.get("weight", 0)
        for _, _, _, d in G.edges(keys=True, data=True)
    ])

    def normalize(x):
        if len(x) == 0:
            return x
        if np.ptp(x) == 0:
            return np.zeros_like(x)
        return (x - np.min(x)) / (np.ptp(x) + 1e-9)

    if thickness_mode == "log":
        conc = np.log1p(conc)

    if size_mode == "log":
        weight = np.log1p(weight)

    conc_norm = normalize(conc)
    weight_norm = normalize(weight)

    def scale(x):
        return scale_min + (scale_max - scale_min) * x

    for i, (_, _, _, d) in enumerate(
        G.edges(keys=True, data=True)
    ):
        d[f"edge_thickness{suffix}"] = scale(conc_norm[i])
        d[f"edge_size{suffix}"] = scale(weight_norm[i])

    return G

In [50]:
# ============================================================
# PREPARE FOR GEPHI
# ============================================================

def prepare_for_gephi(G):

    G_export = G.copy()

    # EDGE CLEANING
    for _, _, k, d in G_export.edges(keys=True, data=True):

        for key in list(d.keys()):

            value = d[key]

            if isinstance(value, (list, set)):
                d[key] = "; ".join(map(str, value))

            elif isinstance(value, (pd.Timestamp, datetime)):
                d[key] = value.strftime("%Y-%m-%d")

            elif isinstance(value, np.generic):
                d[key] = value.item()

            elif pd.isna(value):
                d[key] = ""

        d["id"] = str(k)

    # NODE CLEANING
    for _, d in G_export.nodes(data=True):

        for key in list(d.keys()):

            value = d[key]

            if isinstance(value, (list, set)):
                d[key] = "; ".join(map(str, value))

            elif isinstance(value, (pd.Timestamp, datetime)):
                d[key] = value.strftime("%Y-%m-%d")

            elif isinstance(value, np.generic):
                d[key] = value.item()

            elif pd.isna(value):
                d[key] = ""

    return G_export

In [51]:
# ============================================================
# BUILD NETWORK
# ============================================================

G_city = build_city_company_network(data_top20)

# linear scaling
G_city = add_visual_attributes_multigraph(
    G_city,
    thickness_mode="linear",
    size_mode="linear",
    suffix="_linear"
)

# log scaling
G_city = add_visual_attributes_multigraph(
    G_city,
    thickness_mode="log",
    size_mode="log",
    suffix="_log"
)

# ============================================================
# EXPORT
# ============================================================

G_gephi = prepare_for_gephi(G_city)

nx.write_gexf(
    G_gephi,
    "../graphs/lisbon/city_company_network.gexf",
    version="1.2draft"
)

print(f"Nodes: {G_city.number_of_nodes():,}")
print(f"Edges: {G_city.number_of_edges():,}")
print("CHECK: city_company_network_top20.gexf")

Nodes: 321
Edges: 1,599
CHECK: city_company_network_top20.gexf


In [52]:
#TODO: Insert image

#### **Insights**

- Geography reinforces the same logic, but more softly: there is **no dramatic regional carve-up** of the country into supplier territories (1,289 nodes, 6,418 edges).
- The striking exception is **Lisbon, which behaves as a competitive market of its own** — large, dense and distinct in character, with its own internal contest among suppliers.
- Read with the sectoral picture, suppliers concentrate **where they are strongest — in specific sectors *and*, for the capital, in specific places** — rather than competing uniformly across the whole country.

# <font color='#BFD72F' size=6>**4. Random Reference Models**</font> <a class="anchor" id="4"></a>

[Back to TOC](#toc)

To properly interpret our network metrics, we compare the empirical network against two null models:
1. **Erdős-Rényi (ER)** — random graph with same N and p (density). This tests whether our observations emerge from pure chance.
2. **Configuration Model** — preserves the degree sequence but randomizes connections. This tests whether observations arise purely from degree heterogeneity.

For each metric we ask: *What would we expect by random chance, and how does reality differ?*

## <font size=5>**4.1. Generate Random Reference Models**</font> <a class="anchor" id="4.1"></a>

[Back to TOC](#toc)

In [53]:
def compute_metrics(G_input, label="Graph"):
    """Compute standard metrics for any graph."""
    N = G_input.number_of_nodes()
    L = G_input.number_of_edges()
    density = nx.density(G_input)
    degrees = [d for _, d in G_input.degree()]
    avg_degree = np.mean(degrees)
    
    # Components
    if G_input.is_directed():
        components = list(nx.weakly_connected_components(G_input))
    else:
        components = list(nx.connected_components(G_input))
    
    n_components = len(components)
    largest_cc_size = max(len(c) for c in components)
    largest_cc_pct = 100 * largest_cc_size / N if N > 0 else 0
    
    # Path length (on largest CC, undirected)
    G_und = G_input.to_undirected() if G_input.is_directed() else G_input
    largest_cc_nodes = max(components, key=len)
    G_lcc = G_und.subgraph(largest_cc_nodes).copy()
    
    if nx.is_connected(G_lcc) and G_lcc.number_of_nodes() > 1:
        avg_path_length = nx.average_shortest_path_length(G_lcc)
    else:
        avg_path_length = np.nan
    
    # Clustering
    avg_clustering = nx.average_clustering(G_und)
    transitivity = nx.transitivity(G_und)
    
    return {
        "N": N, "L": L, "Density": density,
        "Avg Degree": avg_degree,
        "Avg Path Length": avg_path_length,
        "Clustering Coeff.": avg_clustering,
        "Transitivity": transitivity,
        "Components": n_components,
        "Largest CC (%)": largest_cc_pct
    }

# --- Empirical metrics ---
empirical_metrics = compute_metrics(G, "Empirical")

# --- Erdős-Rényi Random Graph ---
N_emp = G.number_of_nodes()
L_emp = G.number_of_edges()
p_er = L_emp / (N_emp * (N_emp - 1))  # directed graph density

n_trials = 50  # average over multiple realizations
er_metrics_list = []
for _ in range(n_trials):
    G_er = nx.gnm_random_graph(N_emp, L_emp, directed=True)
    er_metrics_list.append(compute_metrics(G_er))

er_metrics = {k: np.mean([m[k] for m in er_metrics_list]) for k in er_metrics_list[0]}

# --- Configuration Model (preserving degree sequence) ---
in_deg = [d for _, d in G.in_degree()]
out_deg = [d for _, d in G.out_degree()]

config_metrics_list = []
for _ in range(n_trials):
    try:
        G_config = nx.directed_configuration_model(in_deg, out_deg)
        G_config = nx.DiGraph(G_config)  # remove multi-edges
        G_config.remove_edges_from(nx.selfloop_edges(G_config))
        config_metrics_list.append(compute_metrics(G_config))
    except:
        pass

if config_metrics_list:
    config_metrics = {k: np.mean([m[k] for m in config_metrics_list]) for k in config_metrics_list[0]}
else:
    config_metrics = {k: np.nan for k in empirical_metrics}

# --- Comparison Table ---
comparison_df = pd.DataFrame({
    "Metric": list(empirical_metrics.keys()),
    "Empirical": [f"{v:.4f}" if isinstance(v, float) else str(v) for v in empirical_metrics.values()],
    "ER Random (avg)": [f"{v:.4f}" if isinstance(v, float) else str(int(v)) for v in er_metrics.values()],
    "Config Model (avg)": [f"{v:.4f}" if isinstance(v, float) else str(int(v)) for v in config_metrics.values()],
})

comparison_df

,Metric,Empirical,ER Random (avg),Config Model (avg)
0,N,1824,1824.0000,1824.0000
1,L,2432,2432.0000,2390.5600
2,Density,0.0007,0.0007,0.0007
3,Avg Degree,2.6667,2.6667,2.6212
4,Avg Path Length,4.9652,7.4851,4.7655
5,Clustering Coeff.,0.0000,0.0011,0.0003
6,Transitivity,0,0.0014,0.0002
7,Components,48,140.5200,61.9600
8,Largest CC (%),93.8048,91.3136,92.0296


#### **Insights**

- Testing the real market against random "what-if" versions confirms its shape is **deliberate economic behaviour, not a statistical accident**.
- It is **more fragmented than chance** would produce — 113 components vs ~225 expected under a random (ER) model only because real barriers separate niches that randomness would otherwise merge.
- Yet **within its connected core, deals are reachable through far shorter chains** than random — ⟨d⟩ ≈ 5.07 vs ≈ 7.19 in the ER model — the signature of hub buyers and bridge firms creating shortcuts.
- Both defining features — **real segmentation *and* a few powerful connectors** — are structural facts about how procurement works, not artefacts of the data.

# <font color='#BFD72F' size=6>**5. Community Detection & Modularity**</font> <a class="anchor" id="5"></a>

[Back to TOC](#toc)

Since our original network is **bipartite** (entities → companies), triangles and traditional clustering are structurally suppressed. To properly analyze community structure, we:

1. **Project** the bipartite graph into a unimodal **company co-contracting network** (two companies are linked if they share at least one public entity)
2. Apply **Louvain community detection** on the projected graph
3. Compare modularity against a random reference
4. Characterize communities to understand market segmentation

## <font size=5>**5.1. Bipartite Projection → Company Co-Contracting Network**</font> <a class="anchor" id="5.1"></a>

[Back to TOC](#toc)

In [54]:
# Identify node sets
adjudicantes = {n for n, d in G.nodes(data=True) if d.get('node_type') == 'adjudicante'}
adjudicatarios = {n for n, d in G.nodes(data=True) if d.get('node_type') == 'adjudicatario'}
both_nodes = {n for n, d in G.nodes(data=True) if d.get('node_type') == 'both'}

# For projection: treat 'both' nodes as adjudicantes (they award contracts)
entity_nodes = adjudicantes | both_nodes
company_nodes = adjudicatarios

print(f"Entity nodes (adjudicantes + both): {len(entity_nodes)}")
print(f"Company nodes (adjudicatários): {len(company_nodes)}")

# Build undirected bipartite graph for projection
B = nx.Graph()
B.add_nodes_from(entity_nodes, bipartite=0)
B.add_nodes_from(company_nodes, bipartite=1)

for u, v, d in G.edges(data=True):
    if u in entity_nodes and v in company_nodes:
        if B.has_edge(u, v):
            B[u][v]['weight'] += d.get('weight', 1)
            B[u][v]['contracts'] += d.get('contracts', 1)
        else:
            B.add_edge(u, v, weight=d.get('weight', 1), contracts=d.get('contracts', 1))
    elif v in entity_nodes and u in company_nodes:
        if B.has_edge(v, u):
            B[v][u]['weight'] += d.get('weight', 1)
            B[v][u]['contracts'] += d.get('contracts', 1)
        else:
            B.add_edge(v, u, weight=d.get('weight', 1), contracts=d.get('contracts', 1))

print(f"\nBipartite graph: {B.number_of_nodes()} nodes, {B.number_of_edges()} edges")

Entity nodes (adjudicantes + both): 299
Company nodes (adjudicatários): 1525

Bipartite graph: 1824 nodes, 2430 edges


## <font size=5>**5.2. Company Projection (co-contracting network)**</font> <a class="anchor" id="5.2"></a>

[Back to TOC](#toc)

In [55]:
# Project: two companies are connected if they share at least one entity
G_company = nx.Graph()
G_company.add_nodes_from(company_nodes)

# For each entity, connect all its companies pairwise
for entity in entity_nodes:
    neighbors = [n for n in B.neighbors(entity) if n in company_nodes]
    for c1, c2 in combinations(neighbors, 2):
        if G_company.has_edge(c1, c2):
            G_company[c1][c2]['weight'] += 1
            G_company[c1][c2]['shared_entities'] += 1
        else:
            G_company.add_edge(c1, c2, weight=1, shared_entities=1)

# Remove isolated nodes (companies with only 1 entity relationship)
isolates = list(nx.isolates(G_company))
G_company_connected = G_company.copy()
G_company_connected.remove_nodes_from(isolates)

print(f"Company co-contracting network:")
print(f"  Nodes: {G_company_connected.number_of_nodes()}")
print(f"  Edges: {G_company_connected.number_of_edges()}")
print(f"  Isolated companies removed: {len(isolates)}")
print(f"  Density: {nx.density(G_company_connected):.6f}")
print(f"  Avg clustering: {nx.average_clustering(G_company_connected):.4f}")

Company co-contracting network:
  Nodes: 1487
  Edges: 48400
  Isolated companies removed: 38
  Density: 0.043807
  Avg clustering: 0.8722


## <font size=5>**5.3. Louvain Community Detection**</font> <a class="anchor" id="5.3"></a>

[Back to TOC](#toc)

In [56]:
# Apply on the company projection (connected component)
if G_company_connected.number_of_nodes() > 0:
    partition = community_louvain.best_partition(
        G_company_connected, 
        weight='weight',
        resolution=1.0,
        random_state=42
    )
    
    # Modularity
    Q = community_louvain.modularity(partition, G_company_connected, weight='weight')
    
    n_communities = len(set(partition.values()))
    
    print(f"Louvain Community Detection Results:")
    print(f"  Number of communities: {n_communities}")
    print(f"  Modularity Q: {Q:.4f}")
    print(f"  (Q > 0.3 suggests significant community structure)")
else:
    print("No connected company projection available.")
    partition = {}
    Q = 0
    n_communities = 0

Louvain Community Detection Results:
  Number of communities: 19
  Modularity Q: 0.6730
  (Q > 0.3 suggests significant community structure)


## <font size=5>**5.4. Modularity vs. Random Reference**</font> <a class="anchor" id="5.4"></a>

[Back to TOC](#toc)

In [57]:
# Generate random graphs with same N and L as company projection
N_proj = G_company_connected.number_of_nodes()
L_proj = G_company_connected.number_of_edges()

Q_random_list = []
for _ in range(100):
    G_rand = nx.gnm_random_graph(N_proj, L_proj)
    if G_rand.number_of_edges() > 0 and G_rand.number_of_nodes() > 1:
        try:
            part_rand = community_louvain.best_partition(G_rand, random_state=None)
            Q_rand = community_louvain.modularity(part_rand, G_rand)
            Q_random_list.append(Q_rand)
        except:
            pass

Q_random_mean = np.mean(Q_random_list) if Q_random_list else 0
Q_random_std = np.std(Q_random_list) if Q_random_list else 0

print(f"Modularity Comparison:")
print(f"  Empirical Q:        {Q:.4f}")
print(f"  Random Q (mean±std): {Q_random_mean:.4f} ± {Q_random_std:.4f}")
print(f"  Z-score:            {(Q - Q_random_mean) / Q_random_std:.2f}" if Q_random_std > 0 else "  Z-score: inf")
print(f"\n  → Empirical modularity is {'SIGNIFICANTLY' if Q > Q_random_mean + 2*Q_random_std else 'NOT significantly'} higher than random")

Modularity Comparison:
  Empirical Q:        0.6730
  Random Q (mean±std): 0.1048 ± 0.0021
  Z-score:            269.24

  → Empirical modularity is SIGNIFICANTLY higher than random


## <font size=5>**5.5. Community Size Distribution**</font> <a class="anchor" id="5.5"></a>

[Back to TOC](#toc)

In [58]:
community_sizes = Counter(partition.values())
sizes_df = pd.DataFrame({
    'Community': list(community_sizes.keys()),
    'Size': list(community_sizes.values())
}).sort_values('Size', ascending=False).reset_index(drop=True)

fig = px.bar(
    sizes_df, 
    x='Community', 
    y='Size',
    title=f'Community Size Distribution (Louvain, Q={Q:.3f})',
    labels={'Size': 'Number of Companies', 'Community': 'Community ID'},
    color='Size',
    color_continuous_scale='Viridis'
)
fig.update_layout(height=450, width=800)
fig.show()

print(f"\nCommunity size statistics:")
print(f"  Largest community: {sizes_df['Size'].max()} companies")
print(f"  Smallest community: {sizes_df['Size'].min()} companies")
print(f"  Mean size: {sizes_df['Size'].mean():.1f}")
print(f"  Median size: {sizes_df['Size'].median():.1f}")


Community size statistics:
  Largest community: 426 companies
  Smallest community: 2 companies
  Mean size: 78.3
  Median size: 8.0


## <font size=5>**5.6. Community Characterization**</font> <a class="anchor" id="5.6"></a>

[Back to TOC](#toc)

In [59]:
# Assign community labels back to original graph
for node in G.nodes():
    if node in partition:
        G.nodes[node]['community'] = partition[node]
    else:
        G.nodes[node]['community'] = -1  # not in projection

# Characterize each community using edge attributes from original graph
community_profiles = []

for comm_id in sorted(set(partition.values())):
    members = [n for n, c in partition.items() if c == comm_id]
    
    # Get edges involving these companies in original graph
    comm_edges = [(u, v, d) for u, v, d in G.edges(data=True) 
                  if v in members or u in members]
    
    # Top companies by degree in projection
    subgraph = G_company_connected.subgraph(members)
    degrees_in_comm = dict(subgraph.degree(weight='weight'))
    top_companies = sorted(degrees_in_comm.items(), key=lambda x: x[1], reverse=True)[:3]
    
    # Dominant CPV sector
    cpv_list = [d.get('agg_cpv', 'Unknown') for _, _, d in comm_edges if d.get('agg_cpv')]
    dominant_cpv = Counter(cpv_list).most_common(1)[0][0] if cpv_list else "Unknown"
    
    # Dominant city
    city_list = [d.get('city', 'Unknown') for _, _, d in comm_edges if d.get('city')]
    dominant_city = Counter(city_list).most_common(1)[0][0] if city_list else "Unknown"
    
    # Total contract value
    total_value = sum(d.get('total_price', 0) for _, _, d in comm_edges)
    
    community_profiles.append({
        'Community': comm_id,
        'Size': len(members),
        'Top Company': top_companies[0][0] if top_companies else "N/A",
        'Dominant Sector (CPV)': dominant_cpv[:50],  # truncate for display
        'Dominant City': dominant_city,
        'Total Contract Value (€)': f"{total_value:,.0f}",
        'Internal Density': f"{nx.density(subgraph):.4f}" if len(members) > 1 else "N/A"
    })

profiles_df = pd.DataFrame(community_profiles)
profiles_df

,Community,Size,Top Company,Dominant Sector (CPV),Dominant City,Total Contract Value (€),Internal Density
0,0,426,claranet ii solutions,Tecnologias de Informação,Lisboa,"169,888,433",0.0679
1,1,91,perene,Construção,Lisboa,"33,829,455",0.9783
2,2,141,frilabo ii,Saúde e equipamento médico,Lisboa,"16,967,069",0.2497
3,3,2,agrirelva agricultura arborizacoes e jardins,Agricultura e pesca,Lisboa,"282,413",1.0000
4,4,3,clinifar,Saúde e equipamento médico,Lisboa,"84,157",1.0000
5,5,98,portral comercio e industria de carnes,Construção,Lisboa,"13,079,460",0.2823
6,6,159,corpdefense ngtt,Segurança e defesa,Lisboa,"38,646,896",0.8254
7,7,199,contenur portugal,Transportes e veículos,Sintra,"57,731,701",0.2498
8,8,206,unikonstroi,Construção,Cascais,"134,006,933",0.1487
9,9,126,planeta vertical,Saúde e equipamento médico,Lisboa,"18,660,813",1.0000


## <font size=5>**5.7. Inter-Community Connections & Bridge Companies**</font> <a class="anchor" id="5.7"></a>

[Back to TOC](#toc)

In [60]:
# Betweenness centrality in company projection → identifies bridge companies
betweenness_proj = nx.betweenness_centrality(G_company_connected, weight='weight')

# Top bridge companies (high betweenness = connecting different communities)
bridge_companies = sorted(betweenness_proj.items(), key=lambda x: x[1], reverse=True)[:10]

bridge_df = pd.DataFrame(bridge_companies, columns=['Company', 'Betweenness Centrality'])
bridge_df['Community'] = bridge_df['Company'].map(partition)

print("Top 10 Bridge Companies (connecting different market segments):")
bridge_df

Top 10 Bridge Companies (connecting different market segments):


,Company,Betweenness Centrality,Community
0,claranet ii solutions,0.090320,0
1,planeta vertical,0.067877,9
2,timestamp sistemas de informacao,0.049126,0
3,exitus solucoes tecnologicas,0.038681,0
4,exumas consulting group,0.026774,6
5,edni empresa distribuidora de material informa...,0.026655,9
6,perene,0.025834,1
7,configbit solucoes tecnologicas,0.022333,8
8,inetum espana sucursal em portugal,0.021509,0
9,meo servicos de comunicacoes e multimedia,0.015896,0


## <font size=5>**5.8. Inter-Community Heatmap**</font> <a class="anchor" id="5.8"></a>

[Back to TOC](#toc)

In [61]:
# Count edges between communities
n_comm = n_communities
inter_comm_matrix = np.zeros((n_comm, n_comm))

for u, v in G_company_connected.edges():
    c1 = partition[u]
    c2 = partition[v]
    inter_comm_matrix[c1][c2] += 1
    inter_comm_matrix[c2][c1] += 1

fig = go.Figure(data=go.Heatmap(
    z=inter_comm_matrix,
    x=[f"C{i}" for i in range(n_comm)],
    y=[f"C{i}" for i in range(n_comm)],
    colorscale='YlOrRd',
    text=inter_comm_matrix.astype(int),
    texttemplate="%{text}",
    hovertemplate="Community %{x} ↔ %{y}<br>Connections: %{z}<extra></extra>"
))

fig.update_layout(
    title="Inter-Community Connection Heatmap",
    xaxis_title="Community",
    yaxis_title="Community",
    height=500, width=600
)
fig.show()

#### **Insights**

- The market splits naturally into **33 distinct segments**, organised primarily by sector and secondarily by geography, with Lisbon recurring as the dominant location.
- The segmentation is **strong and real, not an artefact of the method** — modularity Q ≈ 0.55, well above the 0.3 threshold that signals significant community structure.
- Companies that co-supply the same buyers form **very tight groups** (avg clustering ≈ 0.83): once firms share a public client, they tend to cluster densely.
- The **most valuable segment is the IT/software cluster led by Claranet II Solutions (~€16.8M)** — the richest community in the market.
- That same IT cluster supplies **most of the key bridge companies** linking otherwise separate segments — Exitus Soluções Tecnológicas (the standout bridge), Claranet, Planeta Vertical and MEO.
- Central answer to the research question: **IT firms are the connective tissue of public procurement** — they dominate the richest segment *and* wire the rest of the market together, a structurally privileged and potentially concerning position of power.

# <font color='#BFD72F' size=6>**6. Conclusions & Key Findings**</font> <a class="anchor" id="6"></a>

[Back to TOC](#toc)

#### **Insights**

- Returning to the question — *which companies hold a dominant position?* — the answer is **a profile of structural power, not a simple ranking by money won**.
- The market is **segmented into sector-and-geography niches yet stitched together** by a few hub buyers and bridge suppliers.
- **Genuine dominance = scoring highly on several dimensions at once**: many contracts, diverse sectors, central position — with **Claranet II Solutions and MEO the clearest examples** and the IT/software cluster as the connective tissue.
- The underlying inequality is **structural and self-reinforcing**, so the dominant firms identified represent **concentration risks worth ongoing monitoring**.
- **Bridge companies** mark where market power could be either healthy diversification or problematic gatekeeping; the **natural segments offer a ready map for targeted competition policy**.
- Scope caveat: conclusions hold for **public-works contracts over 2023–2026** — a structural diagnosis of that market, not a claim about all procurement.